# ZeroMRG Gate3 controlled recovery: epochs91–95 only

One explicitly authorized private session. Restores the independently verified physical90 baseline and full SGD/RNG. No image inference or test scoring. Historical85/56.44423922571333 and inherited stopping remain independent. Strict validation selection saves real recovery-window weights even below the missing85 score. Do not rerun or extend this notebook without separate authorization.

In [ ]:
"""Checkpoint-producing Kaggle decoder and fail-closed recovery utilities.

This profile preserves the recorded COV-CTR experiment, not modular defaults.
No training, model download, or inference is performed on import.
"""
from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import random
import re
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F

ARCHITECTURE = dict(vocab_size=368, max_tokens=160, d_model=768,
                    num_layers=6, num_heads=8, ffn_dim=3072, dropout=0.1,
                    activation="gelu", norm_first=False, final_layer_norm=False,
                    scale_token_embeddings=False, pad_id=0, bos_id=1, eos_id=2)
TRAINING = dict(seed=42, batch_size=6, learning_rate=0.001, momentum=0.99,
                gradient_clip=5.0, patience=50, max_epochs=200,
                precision="float32", scheduler=None,
                selection_metric="validation_text_reconstruction_BLEU4_PTB_x100")
MCLIP_NAME = "M-CLIP/XLM-Roberta-Large-Vit-L-14"
DEPENDENCIES = ("train_manifest.csv", "validation_manifest.csv", "test_manifest.csv",
                "prompt_manifest.csv", "validated_manifest.csv", "validation_audit.json",
                "decoder_vocabulary.json", "mclip_text_features.pt", "mclip_prompt_bank.pt")


def require(condition, message):
    if not condition:
        raise ValueError(message)


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def load_checkpoint(path):
    require(Path(path).is_file(), f"Selected checkpoint does not exist: {path}")
    # No unsafe unpickling fallback. Legacy project artifacts load with weights_only=True.
    return torch.load(path, map_location="cpu", weights_only=True)


class CheckpointReportDecoder(nn.Module):
    def __init__(self, architecture=None):
        super().__init__()
        self.architecture = dict(ARCHITECTURE if architecture is None else architecture)
        a = self.architecture
        require(not a["scale_token_embeddings"] and not a["final_layer_norm"],
                "This class implements the recorded Kaggle decoder only")
        self.token_embedding = nn.Embedding(a["vocab_size"], a["d_model"], padding_idx=a["pad_id"])
        self.position_embedding = nn.Embedding(a["max_tokens"], a["d_model"])
        layer = nn.TransformerDecoderLayer(
            d_model=a["d_model"], nhead=a["num_heads"], dim_feedforward=a["ffn_dim"],
            dropout=a["dropout"], activation=a["activation"], batch_first=True,
            norm_first=a["norm_first"])
        self.transformer = nn.TransformerDecoder(layer, num_layers=a["num_layers"])
        self.output_projection = nn.Linear(a["d_model"], a["vocab_size"])

    def forward(self, condition, input_ids):
        a = self.architecture
        require(input_ids.ndim == 2 and 0 < input_ids.shape[1] <= a["max_tokens"],
                "Decoder IDs must be [B,T] within the saved positional limit")
        require(tuple(condition.shape) == (input_ids.shape[0], a["d_model"]),
                "Condition dimensions incompatible with checkpoint decoder")
        positions = torch.arange(input_ids.shape[1], device=input_ids.device).unsqueeze(0)
        target = self.token_embedding(input_ids) + self.position_embedding(positions)
        mask = torch.triu(torch.ones(input_ids.shape[1], input_ids.shape[1],
                                    dtype=torch.bool, device=input_ids.device), diagonal=1)
        hidden = self.transformer(target, condition.unsqueeze(1).to(target.dtype),
                                  tgt_mask=mask, tgt_key_padding_mask=input_ids.eq(a["pad_id"]))
        return self.output_projection(hidden)

    @torch.no_grad()
    def generate(self, condition, max_length=None):
        self.eval()
        a = self.architecture
        limit = a["max_tokens"] if max_length is None else int(max_length)
        require(2 <= limit <= a["max_tokens"], "Invalid generation length")
        ids = torch.full((len(condition), 1), a["bos_id"], dtype=torch.long, device=condition.device)
        finished = torch.zeros(len(condition), dtype=torch.bool, device=condition.device)
        for _ in range(limit - 1):
            next_id = self(condition, ids)[:, -1].argmax(-1)
            next_id = torch.where(finished, torch.full_like(next_id, a["pad_id"]), next_id)
            ids = torch.cat((ids, next_id[:, None]), dim=1)
            finished |= next_id.eq(a["eos_id"])
            if bool(finished.all()):
                break
        return ids


def validate_model_state(checkpoint, model):
    require(isinstance(checkpoint, dict), "Checkpoint must be a state dictionary payload")
    state = checkpoint.get("decoder")
    require(isinstance(state, dict), "Required decoder state dictionary absent")
    expected = model.state_dict()
    require(set(state) == set(expected),
            f"Decoder key mismatch: missing={sorted(set(expected)-set(state))}; unexpected={sorted(set(state)-set(expected))}")
    for key, tensor in state.items():
        require(isinstance(tensor, torch.Tensor) and tensor.shape == expected[key].shape,
                f"Decoder shape mismatch: {key}")
        require(tensor.dtype == expected[key].dtype, f"Decoder dtype mismatch: {key}")
        require(bool(torch.isfinite(tensor).all()), f"Non-finite decoder weights: {key}")
    if "architecture" in checkpoint:
        require(checkpoint["architecture"] == model.architecture, "Saved architecture metadata differs")
    for key, expected_value in (("vocabulary_size", model.architecture["vocab_size"]),
                                ("max_tokens", model.architecture["max_tokens"])):
        if key in checkpoint:
            require(checkpoint[key] == expected_value, f"Saved {key} differs")


def inspect_training_state(checkpoint, model, best_checkpoint=None):
    require(isinstance(checkpoint, dict), "Training checkpoint must be a dictionary payload")
    required = {"epoch", "decoder", "optimizer", "history", "best_validation_bleu4", "epochs_without_improvement"}
    require(required <= checkpoint.keys(), f"Required training state absent: {sorted(required-checkpoint.keys())}")
    epoch = checkpoint["epoch"]
    require(type(epoch) is int and epoch > 0, "Saved epoch must be a positive integer")
    require(checkpoint.get("scheduler") is None or checkpoint.get("scheduler") == {},
            "Unknown scheduler state; supply its verified implementation before resuming")
    if "training_config" in checkpoint:
        require(checkpoint["training_config"] == TRAINING, "Training configuration differs from recorded run")
    history = checkpoint["history"]
    require(isinstance(history, list) and len(history) == epoch, "History length differs from saved epoch")
    require(all(isinstance(row, dict) and {"epoch", "training_loss", "validation_bleu4"} <= row.keys()
                for row in history), "History record fields absent or invalid")
    require([row["epoch"] for row in history] == list(range(1, epoch + 1)), "History is not contiguous")
    for row in history:
        require(all(math.isfinite(float(row[key])) for key in ("training_loss", "validation_bleu4")),
                "Non-finite history value")
    selected = max(history, key=lambda row: row["validation_bleu4"])
    require(math.isclose(float(selected["validation_bleu4"]), float(checkpoint["best_validation_bleu4"]),
                         rel_tol=0, abs_tol=1e-10), "History and saved best metric disagree")
    require(type(checkpoint["epochs_without_improvement"]) is int and
            checkpoint["epochs_without_improvement"] == epoch - selected["epoch"], "Patience state disagrees with history")
    validate_model_state(checkpoint, model)
    if best_checkpoint is not None:
        validate_selected_best(checkpoint, model, best_checkpoint)
    opt = checkpoint["optimizer"]
    require(isinstance(opt, dict) and {"state", "param_groups"} <= opt.keys(), "Invalid optimizer state")
    require(len(opt["param_groups"]) == 1, "Expected one decoder SGD parameter group")
    group = opt["param_groups"][0]
    for key, expected in dict(lr=0.001, momentum=0.99, dampening=0, weight_decay=0,
                              nesterov=False, maximize=False).items():
        require(group.get(key, False if key == "maximize" else None) == expected,
                f"SGD setting mismatch: {key}")
    parameters = list(model.named_parameters())
    saved_ids = group["params"]
    require(len(saved_ids) == len(parameters) and len(set(saved_ids)) == len(saved_ids),
            "Optimizer parameter count or mapping incompatible")
    require(set(opt["state"]) == set(saved_ids), "Missing or unexpected optimizer parameter states")
    nonzero = 0
    for pid, (name, parameter) in zip(saved_ids, parameters):
        require(set(opt["state"][pid]) == {"momentum_buffer"}, f"Unknown/missing SGD state: {name}")
        buffer = opt["state"][pid]["momentum_buffer"]
        require(isinstance(buffer, torch.Tensor) and buffer.shape == parameter.shape and
                buffer.dtype == parameter.dtype, f"Momentum shape/dtype incompatible: {name}")
        require(bool(torch.isfinite(buffer).all()), f"Non-finite momentum: {name}")
        nonzero += int(bool(buffer.ne(0).any()))
    require(nonzero > 0, "All SGD momentum is zero")
    if "parameter_names" in checkpoint:
        require(checkpoint["parameter_names"] == [name for name, _ in parameters], "Optimizer parameter order differs")
    return dict(epoch=epoch, next_epoch=epoch + 1, best_epoch=selected["epoch"],
                best_validation_bleu4=float(selected["validation_bleu4"]),
                epochs_without_improvement=checkpoint["epochs_without_improvement"],
                momentum_buffers=len(saved_ids), nonzero_momentum_buffers=nonzero,
                parameter_count=sum(p.numel() for _, p in parameters),
                legacy_rng_missing="rng_state" not in checkpoint,
                scheduler="none", compatibility="PASS")


def validate_selected_best(checkpoint, model, best_checkpoint):
    require(isinstance(best_checkpoint, dict), "Selected best must be a dictionary payload")
    selected = max(checkpoint["history"], key=lambda row: row["validation_bleu4"])
    require(best_checkpoint.get("epoch") == selected["epoch"] and
            math.isclose(float(best_checkpoint.get("validation_bleu4", float("nan"))),
                         float(selected["validation_bleu4"]), rel_tol=0, abs_tol=1e-10),
            f"Best checkpoint disagrees with history: need epoch {selected['epoch']}, "
            f"found epoch {best_checkpoint.get('epoch')}. Recover matching weights; do not substitute latest weights.")
    validate_model_state(best_checkpoint, model)


def restore_weights_optimizer(checkpoint, model, device="cpu"):
    """Restore latest state for diagnostics; not authorization to continue training."""
    result = inspect_training_state(checkpoint, model)
    model.load_state_dict(checkpoint["decoder"], strict=True)
    model.to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.99)
    optimizer.load_state_dict(checkpoint["optimizer"])
    for parameter, state in optimizer.state.items():
        state["momentum_buffer"] = state["momentum_buffer"].to(parameter.device)
    for pid, parameter in zip(checkpoint["optimizer"]["param_groups"][0]["params"], model.parameters()):
        require(torch.equal(optimizer.state[parameter]["momentum_buffer"].cpu(),
                            checkpoint["optimizer"]["state"][pid]["momentum_buffer"].cpu()),
                "Momentum changed during restoration")
    model.eval()
    return optimizer, result


def restore_training(checkpoint, model, best_checkpoint, device="cpu"):
    result = inspect_training_state(checkpoint, model, best_checkpoint)
    optimizer, _ = restore_weights_optimizer(checkpoint, model, device)
    return optimizer, result


def read_manifest(directory, name):
    with (Path(directory) / name).open(encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))


def validate_artifacts(directory, expected_hashes=None):
    directory = Path(directory)
    hashes = {name: sha256(directory / name) for name in DEPENDENCIES}
    if expected_hashes is not None:
        require(hashes == expected_hashes, "Feature/vocabulary/split dependency hashes differ")
    records = {name: read_manifest(directory, name + "_manifest.csv")
               for name in ("train", "validation", "test", "prompt", "validated")}
    ids = {}
    for name, count in dict(train=571, validation=71, test=72, prompt=500, validated=714).items():
        rows = records[name]
        ids[name] = [row["image_id"] for row in rows]
        require(len(rows) == count and len(set(ids[name])) == count, f"{name} count/ID uniqueness failed")
    train, val, test = (set(ids[name]) for name in ("train", "validation", "test"))
    require(not (train & val or train & test or val & test), "Split leakage")
    require(train | val | test == set(ids["validated"]), "Split population differs")
    require(set(ids["prompt"]) <= train, "Prompt bank contains non-training IDs")
    by_id = {r["image_id"]: r for r in records["validated"]}
    for name in ("train", "validation", "test", "prompt"):
        require(all(row["report"] == by_id[row["image_id"]]["report"] for row in records[name]),
                f"{name} reports differ from validated manifest")
    audit = json.loads((directory / "validation_audit.json").read_text(encoding="utf-8"))
    require(audit["csv_rows"] == 726 and audit["physical_images"] == 746, "Raw audit population differs")
    vocabulary = json.loads((directory / "decoder_vocabulary.json").read_text(encoding="utf-8"))
    require(vocabulary["max_tokens"] == 160 and len(vocabulary["vocabulary"]) == 368,
            "Vocabulary or positional capacity differs")
    require(vocabulary["special_token_ids"] == dict(pad=0, bos=1, eos=2, unk=3), "Special-token IDs differ")
    expected_vocab = ["<pad>", "<bos>", "<eos>", "<unk>"] + sorted({
        token for row in records["train"] for token in re.findall(r"\w+|[^\w\s]", row["report"])})
    require(vocabulary["vocabulary"] == expected_vocab, "Vocabulary is not the saved training-only lexicographic mapping")
    text = load_checkpoint(directory / "mclip_text_features.pt")
    require(text["model"] == MCLIP_NAME and text["normalization"] == "l2", "Text encoder metadata differs")
    for name in ("train", "validation"):
        require(set(text[name]) == set(ids[name]), f"{name} feature membership differs")
        for vector in text[name].values():
            require(vector.shape == (768,) and bool(torch.isfinite(vector).all()) and
                    torch.allclose(vector.norm(), torch.tensor(1.0), atol=1e-4), "Invalid normalized text feature")
    prompt = load_checkpoint(directory / "mclip_prompt_bank.pt")
    require(prompt["model"] == MCLIP_NAME and prompt.get("normalization") == "raw", "Prompt encoder/normalization differs")
    require(prompt["prompt_ids"] == ids["prompt"], "Prompt tensor row order/provenance differs")
    vectors = prompt["embeddings"]
    require(vectors.shape == (500, 768) and bool(torch.isfinite(vectors).all()) and
            bool((vectors.norm(dim=-1) > 0).all()), "Invalid prompt bank")
    aligned = torch.stack([text["train"][key] for key in ids["prompt"]])
    # Separate encodings are not bitwise equal. Record the observed difference;
    # membership and frozen-cache hashes above are the provenance assertions.
    normalized_prompts = F.normalize(vectors, dim=-1)
    cosine = (normalized_prompts * aligned).sum(-1)
    cache_comparison = dict(max_absolute_difference=float((normalized_prompts - aligned).abs().max()),
                            minimum_cosine=float(cosine.min()), mean_cosine=float(cosine.mean()),
                            exact_encoder_revision="UNVERIFIED")
    return dict(records=records, vocabulary=vocabulary, text=text, prompt=prompt,
                dependency_hashes=hashes, prompt_text_comparison=cache_comparison)


def encode_report(text, vocabulary):
    lookup = {token: i for i, token in enumerate(vocabulary)}
    ids = [1] + [lookup.get(token, 3) for token in re.findall(r"\w+|[^\w\s]", text)] + [2]
    require(len(ids) <= 160, "Report exceeds saved positional capacity; no silent truncation")
    return ids


def decode_report(token_ids, vocabulary):
    tokens = []
    for raw in token_ids:
        token = int(raw)
        if token == 2:
            break
        require(0 <= token < len(vocabulary), "Generated token outside vocabulary")
        if token not in {0, 1}:
            tokens.append(vocabulary[token])
    text = re.sub(r"\s+([.,!?;:%)\]])", r"\1", " ".join(tokens))
    return re.sub(r"([(\[])\s+", r"\1", text).strip()


def enrich(image, prompts):
    require(image.ndim == prompts.ndim == 2 and image.shape[-1] == prompts.shape[-1] == 768,
            "Image/prompt dimensions must share the recorded 768-D CLIP space")
    require(bool(torch.isfinite(image).all()) and bool(torch.isfinite(prompts).all()), "Non-finite encoder output")
    scores = image @ prompts.T / math.sqrt(768)
    attention = scores.softmax(-1)
    residual = image + attention @ prompts
    require(bool((residual.norm(dim=-1) > 0).all()), "Zero residual embedding")
    phi = F.normalize(residual, p=2, dim=-1)
    require(bool(torch.isfinite(phi).all()), "Non-finite enriched features")
    return phi, attention


def capture_rng():
    return dict(python=random.getstate(), torch_cpu=torch.get_rng_state(),
                torch_cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [])


def restore_rng(state):
    random.setstate(state["python"])
    torch.set_rng_state(state["torch_cpu"].cpu())
    if state["torch_cuda"]:
        require(len(state["torch_cuda"]) == torch.cuda.device_count(), "CUDA RNG device count differs")
        torch.cuda.set_rng_state_all([value.cpu() for value in state["torch_cuda"]])


def atomic_save(payload, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".tmp")
    with temporary.open("wb") as handle:
        torch.save(payload, handle)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, destination)


def training_payload(model, optimizer, epoch, history, best_metric, bad_epochs, hashes, provenance):
    return dict(schema_version="zeromrg_covctr_gate1_v1", epoch=epoch, decoder=model.state_dict(),
                optimizer=optimizer.state_dict(), scheduler=None, history=history,
                best_validation_bleu4=best_metric, epochs_without_improvement=bad_epochs,
                architecture=model.architecture, parameter_names=[n for n, _ in model.named_parameters()],
                training_config=TRAINING, dependency_hashes=hashes, provenance=provenance,
                rng_state=capture_rng())

In [ ]:
"""Separate recovery selection with historical stopping; no execution on import."""
from __future__ import annotations

import copy
import hashlib
import json
import math
import random
from pathlib import Path

import torch


ORIGINAL_FILES = {
    "text_decoder_last.pt": "ecc204f031b0fa83b4734c3228f20ba2172ce291288f31a6551921126546c299",
    "text_decoder_best.pt": "dba64646e89f1c55c69c9e3a7361141829e4dc7ec32114f98f2edce8b89a0568",
    "text_training_history.json": "ccd65f2ec0ce2f569f683233f5a13c48c26bb27075aa9661bc7a272c3a14a015",
}
ORIGINAL_RECORD = dict(epoch=90, best_epoch=85, best_validation_bleu4=56.44423922571333,
                       epochs_without_improvement=5, patience=50,
                       selected_weights_available=False, stale_best_epoch=8,
                       files=ORIGINAL_FILES)
RECOVERY_POLICY = dict(id="A_historical_stop_available_best_v1", baseline_epoch=90,
                       selection="strict_validation_BLEU4_improvement_in_epochs_90_onward",
                       stopping="original_history_plus_continuation_patience_50",
                       max_epoch=200, max_epochs_per_session=5,
                       historical_weights_missing=True,
                       deviation="selection_restricted_to_recoverable_window; legacy_RNG_lost")
RECOVERY_SCHEMA = "zeromrg_gate3_dual_ledger_v1"


def plain_metadata(value):
    """Normalize str subclasses (including TorchVersion) into safe plain types."""
    return json.loads(json.dumps(value, allow_nan=False))


def json_sha256(value):
    raw = json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()
    return hashlib.sha256(raw).hexdigest()


def capture_recovery_rng():
    import numpy as np
    result = capture_rng()
    name, keys, position, has_gauss, cached = np.random.get_state()
    result["numpy"] = dict(name=name, keys=keys.tolist(), position=int(position),
                            has_gauss=int(has_gauss), cached=float(cached))
    return result


def restore_recovery_rng(state):
    import numpy as np
    require(isinstance(state, dict) and {"python", "torch_cpu", "torch_cuda", "numpy"} <= state.keys(),
            "Required recovery RNG state absent")
    restore_rng(state)
    value = state["numpy"]
    np.random.set_state((value["name"], np.asarray(value["keys"], dtype=np.uint32),
                         value["position"], value["has_gauss"], value["cached"]))


def seed_legacy_recovery():
    import numpy as np
    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)


def historical_stop_record(original_history, continuation_history):
    history = original_history + continuation_history
    selected = max(history, key=lambda row: row["validation_bleu4"])
    return dict(best_epoch=selected["epoch"], best_validation_bleu4=float(selected["validation_bleu4"]),
                epochs_without_improvement=history[-1]["epoch"] - selected["epoch"], patience=50)


def safe_name(name):
    require(isinstance(name, str) and name and Path(name).name == name and
            name not in {".", ".."} and "/" not in name and "\\" not in name,
            "Artifact reference must be a relative basename")
    return name


def valid_digest(value):
    return isinstance(value, str) and len(value) == 64 and all(c in "0123456789abcdef" for c in value)


def validate_measurement(row):
    required = {"epoch", "stage", "validation_bleu4", "predictions_file", "predictions_sha256",
                "metric_fingerprint_sha256"}
    require(isinstance(row, dict) and required <= row.keys(), "Validation measurement fields absent")
    require(type(row["epoch"]) is int and row["epoch"] >= 90, "Invalid measured epoch")
    require(math.isfinite(float(row["validation_bleu4"])) and 0 <= float(row["validation_bleu4"]) <= 100,
            "Invalid measured BLEU4")
    safe_name(row["predictions_file"])
    require(valid_digest(row["predictions_sha256"]) and valid_digest(row["metric_fingerprint_sha256"]),
            "Invalid measurement integrity digest")


def new_recovery_metadata(original_history, baseline, recovery_id, hashes, metric_fingerprint, provenance):
    validate_measurement(baseline)
    require(baseline["epoch"] == 90 and baseline["stage"] == "baseline", "Must measure epoch90 baseline first")
    require(len(original_history) == 90, "Original history must contain exactly90 epochs")
    require(baseline["metric_fingerprint_sha256"] == json_sha256(metric_fingerprint), "Baseline metric identity differs")
    return dict(schema_version=RECOVERY_SCHEMA, epoch=90, recovery_id=recovery_id,
                recovery_policy=copy.deepcopy(RECOVERY_POLICY), historical_record=copy.deepcopy(ORIGINAL_RECORD),
                original_history=copy.deepcopy(original_history), continuation_history=[],
                recovery_history=[copy.deepcopy(baseline)],
                original_stop_tracking=historical_stop_record(original_history, []),
                dependency_hashes=copy.deepcopy(hashes), metric_fingerprint=plain_metadata(metric_fingerprint),
                provenance=plain_metadata(provenance))


def append_recovery_epoch(metadata, epoch, training_loss, measurement):
    """Pure metadata update; no optimizer/model operation and no historical mutation."""
    validate_measurement(measurement)
    require("decoder" not in metadata and "optimizer" not in metadata, "Pass metadata only, not model/optimizer tensors")
    require(epoch == metadata["epoch"] + 1 and measurement["epoch"] == epoch and
            measurement["stage"] == "continuation", "Continuation epoch is not the next completed epoch")
    require(epoch <= 200 and metadata["original_stop_tracking"]["epochs_without_improvement"] < 50,
            "Original stopping condition already met")
    require(math.isfinite(float(training_loss)), "Non-finite training loss")
    require(measurement["metric_fingerprint_sha256"] == json_sha256(metadata["metric_fingerprint"]),
            "Metric implementation changed")
    current_best = max(metadata["recovery_history"], key=lambda row: row["validation_bleu4"])
    improved = float(measurement["validation_bleu4"]) > float(current_best["validation_bleu4"])
    result = copy.deepcopy(metadata)
    # This function accepts metadata only, never a payload containing model tensors.
    result["epoch"] = epoch
    result["continuation_history"].append(dict(epoch=epoch, training_loss=float(training_loss),
                                               validation_bleu4=float(measurement["validation_bleu4"])))
    result["recovery_history"].append(copy.deepcopy(measurement))
    result["original_stop_tracking"] = historical_stop_record(result["original_history"], result["continuation_history"])
    return result, improved


def recovery_best_payload(model, measurement, recovery_id, dependency_hashes):
    validate_measurement(measurement)
    return dict(schema_version="zeromrg_gate3_recovery_best_v1", decoder=model.state_dict(),
                epoch=measurement["epoch"], validation_bleu4=float(measurement["validation_bleu4"]),
                validation_predictions_sha256=measurement["predictions_sha256"],
                metric_fingerprint_sha256=measurement["metric_fingerprint_sha256"],
                recovery_id=recovery_id, selection_scope="recovery_epochs_90_onward",
                historical_selected_epoch=85, is_missing_epoch85_model=False,
                architecture=model.architecture, vocabulary_size=model.architecture["vocab_size"],
                max_tokens=model.architecture["max_tokens"], dependency_hashes=dependency_hashes)


def exclusive_save(payload, path):
    require(not Path(path).exists(), "Immutable artifact already exists; refuse overwrite")
    atomic_save(payload, path)


def best_reference(path, measurement):
    return dict(file=safe_name(Path(path).name), sha256=sha256(path), epoch=measurement["epoch"],
                validation_bleu4=float(measurement["validation_bleu4"]),
                predictions_file=safe_name(measurement["predictions_file"]),
                predictions_sha256=measurement["predictions_sha256"])


def latest_recovery_payload(metadata, model, optimizer, reference):
    result = copy.deepcopy(metadata)
    result.update(decoder=model.state_dict(), optimizer=optimizer.state_dict(), scheduler=None,
                  architecture=model.architecture, parameter_names=[n for n, _ in model.named_parameters()],
                  training_config=TRAINING, recovery_best=copy.deepcopy(reference), rng_state=capture_recovery_rng())
    return result


def metadata_from_payload(payload):
    return {key: copy.deepcopy(payload[key]) for key in
            ("schema_version", "epoch", "recovery_id", "recovery_policy", "historical_record", "original_history",
             "continuation_history", "recovery_history", "original_stop_tracking", "dependency_hashes",
             "metric_fingerprint", "provenance")}


def legacy_view(payload):
    stop = payload["original_stop_tracking"]
    return dict(epoch=payload["epoch"], decoder=payload["decoder"], optimizer=payload["optimizer"],
                history=payload["original_history"] + payload["continuation_history"],
                best_validation_bleu4=stop["best_validation_bleu4"],
                epochs_without_improvement=stop["epochs_without_improvement"],
                scheduler=payload["scheduler"], architecture=payload["architecture"],
                parameter_names=payload["parameter_names"], training_config=payload["training_config"],
                rng_state=payload["rng_state"])


def validate_recovery_state(payload, model, original_history, hashes, best=None):
    require(isinstance(payload, dict), "Recovery checkpoint must be a dictionary")
    required = {"schema_version", "epoch", "recovery_id", "recovery_policy", "historical_record", "original_history",
                "continuation_history", "recovery_history", "original_stop_tracking", "dependency_hashes",
                "metric_fingerprint", "provenance", "decoder", "optimizer", "scheduler", "architecture",
                "parameter_names", "training_config", "recovery_best", "rng_state"}
    require(required <= payload.keys(), "Required recovery state absent: " + str(sorted(required - payload.keys())))
    require(payload["schema_version"] == RECOVERY_SCHEMA and payload["recovery_policy"] == RECOVERY_POLICY,
            "Unknown or altered recovery policy")
    require(payload["historical_record"] == ORIGINAL_RECORD and payload["original_history"] == original_history,
            "Original historical evidence changed")
    require(payload["dependency_hashes"] == hashes, "Split/feature/vocabulary dependencies changed")
    require(payload["scheduler"] is None, "This recovery protocol has no scheduler")
    epoch = payload["epoch"]
    require(type(epoch) is int and 90 <= epoch <= 200, "Recovery epoch outside approved protocol")
    continuation = payload["continuation_history"]
    require(isinstance(continuation, list) and all(isinstance(row, dict) and
            {"epoch", "training_loss", "validation_bleu4"} <= row.keys() for row in continuation),
            "Continuation history fields absent")
    require(len(original_history) == 90 and len(continuation) == epoch - 90, "Historical/continuation lengths disagree")
    require([row["epoch"] for row in continuation] == list(range(91, epoch + 1)), "Continuation history is not contiguous")
    tracking = historical_stop_record(original_history, continuation)
    require(payload["original_stop_tracking"] == tracking, "Original stopping ledger changed/reset")
    for index in range(len(continuation)):
        require(historical_stop_record(original_history, continuation[:index])["epochs_without_improvement"] < 50,
                "Continuation advanced beyond original early stopping")
    rows = payload["recovery_history"]
    require(len(rows) == epoch - 89 and [r["epoch"] for r in rows] == list(range(90, epoch + 1)),
            "Recovery selection history is not contiguous")
    for index, row in enumerate(rows):
        validate_measurement(row)
        require(row["stage"] == ("baseline" if index == 0 else "continuation"), "Recovery measurement stage differs")
        require(row["metric_fingerprint_sha256"] == json_sha256(payload["metric_fingerprint"]),
                "Validation metric fingerprints disagree")
        if index:
            require(float(row["validation_bleu4"]) == float(continuation[index - 1]["validation_bleu4"]),
                    "Validation score differs between ledgers")
    selected = max(rows, key=lambda row: row["validation_bleu4"])
    reference = payload["recovery_best"]
    require(isinstance(reference, dict) and {"file", "sha256", "epoch", "validation_bleu4", "predictions_file",
                                          "predictions_sha256"} <= reference.keys(), "Recovery best reference fields absent")
    safe_name(reference["file"])
    require(valid_digest(reference["sha256"]), "Invalid selected-weight checksum")
    require(reference["epoch"] == selected["epoch"] and reference["validation_bleu4"] == selected["validation_bleu4"] and
            reference["predictions_file"] == selected["predictions_file"] and
            reference["predictions_sha256"] == selected["predictions_sha256"], "Recovery best does not match measured validation history")
    require(isinstance(payload["rng_state"], dict) and
            {"python", "torch_cpu", "torch_cuda", "numpy"} <= payload["rng_state"].keys(), "Recovery RNG state absent")
    result = inspect_training_state(legacy_view(payload), model)
    if best is not None:
        require(best.get("schema_version") == "zeromrg_gate3_recovery_best_v1" and
                best.get("recovery_id") == payload["recovery_id"] and best.get("epoch") == selected["epoch"] and
                best.get("validation_bleu4") == selected["validation_bleu4"] and
                best.get("validation_predictions_sha256") == selected["predictions_sha256"] and
                best.get("metric_fingerprint_sha256") == selected["metric_fingerprint_sha256"] and
                best.get("dependency_hashes") == hashes and best.get("selection_scope") == "recovery_epochs_90_onward" and
                best.get("is_missing_epoch85_model") is False and best.get("historical_selected_epoch") == 85,
                "Selected recovery weights have invalid provenance/score")
        validate_model_state(best, model)
    result.update(recovery_best_epoch=selected["epoch"], recovery_best_validation_bleu4=selected["validation_bleu4"],
                  recovery_non_improvement_counter=epoch - selected["epoch"],
                  historical_rng_missing=True,
                  stop=tracking["epochs_without_improvement"] >= 50 or epoch >= 200)
    return result


def validate_recovery_files(payload, model, original_history, hashes, directory, validation_ids):
    directory = Path(directory)
    ref = payload["recovery_best"]
    best_path = directory / safe_name(ref["file"])
    require(sha256(best_path) == ref["sha256"], "Selected recovery weights checksum mismatch")
    best = load_checkpoint(best_path)
    result = validate_recovery_state(payload, model, original_history, hashes, best)
    for row in payload["recovery_history"]:
        path = directory / safe_name(row["predictions_file"])
        require(sha256(path) == row["predictions_sha256"], "Validation predictions checksum mismatch")
        prediction = json.loads(path.read_text(encoding="utf-8"))
        require(isinstance(prediction, dict) and {"split", "image_ids", "epoch", "validation_bleu4",
                "metric_fingerprint_sha256", "references", "predictions"} <= prediction.keys(),
                "Validation prediction fields absent")
        require(prediction["split"] == "validation" and prediction["image_ids"] == validation_ids,
                "Validation predictions contain wrong IDs/split; possible leakage")
        require(len(prediction["references"]) == len(validation_ids) == len(prediction["predictions"]) and
                all(isinstance(text, str) for text in prediction["references"] + prediction["predictions"]),
                "Validation references/predictions length or types differ")
        require(prediction["epoch"] == row["epoch"] and prediction["validation_bleu4"] == row["validation_bleu4"] and
                prediction["metric_fingerprint_sha256"] == row["metric_fingerprint_sha256"], "Saved validation measurement differs")
    return result


def restore_recovery(payload, model, original_history, hashes, directory, validation_ids, device="cpu"):
    result = validate_recovery_files(payload, model, original_history, hashes, directory, validation_ids)
    optimizer, _ = restore_weights_optimizer(legacy_view(payload), model, device)
    # RNG restored again immediately before training, after any diagnostics/evaluation.
    restore_recovery_rng(payload["rng_state"])
    return optimizer, result

In [ ]:
CANONICAL_DEPENDENCY_HASHES = {'train_manifest.csv': '8902b8fea1163429aec6940a9744f7c792d1b7f9a00ca300a1fab17448ead973', 'validation_manifest.csv': '94c555cb25a19485616065079139722bfe7122a17639267ba5e03d9e4cfa7c5d', 'test_manifest.csv': 'fcbcd1bd12220b0a3a286c4b1a82a8543e34c0a3495c5d0b8c89c81033b3eb66', 'prompt_manifest.csv': 'f60c45a1dc9beecba16917d767f1cf6b0f67231141a9226efdb771a27f9281de', 'validated_manifest.csv': '1536c6eb78e2857e599da4f7b3854ed5fa01fd66e50f390dd315d72fe5572aee', 'validation_audit.json': 'f3706a7197e47c1a2693464f9f01452072ac9e6084388966968e37b42d0c3fae', 'decoder_vocabulary.json': 'c0e169c3558fe0cbf2e36431281aa223f64e2e4fb88f7361e23b33e12bcc3ad5', 'mclip_text_features.pt': '7275ebb5ea1a9ef273eee38b16e45fe6a3f42f154bb844ca2ba890e09853863a', 'mclip_prompt_bank.pt': '3cb770a2a470c70d95952828cf23abc088250571897b960a2a39e9bae6c8e1ce'}
RUNTIME_SOURCE = '"""Checkpoint-producing Kaggle decoder and fail-closed recovery utilities.\n\nThis profile preserves the recorded COV-CTR experiment, not modular defaults.\nNo training, model download, or inference is performed on import.\n"""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nimport math\nimport os\nimport random\nimport re\nfrom pathlib import Path\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nARCHITECTURE = dict(vocab_size=368, max_tokens=160, d_model=768,\n                    num_layers=6, num_heads=8, ffn_dim=3072, dropout=0.1,\n                    activation="gelu", norm_first=False, final_layer_norm=False,\n                    scale_token_embeddings=False, pad_id=0, bos_id=1, eos_id=2)\nTRAINING = dict(seed=42, batch_size=6, learning_rate=0.001, momentum=0.99,\n                gradient_clip=5.0, patience=50, max_epochs=200,\n                precision="float32", scheduler=None,\n                selection_metric="validation_text_reconstruction_BLEU4_PTB_x100")\nMCLIP_NAME = "M-CLIP/XLM-Roberta-Large-Vit-L-14"\nDEPENDENCIES = ("train_manifest.csv", "validation_manifest.csv", "test_manifest.csv",\n                "prompt_manifest.csv", "validated_manifest.csv", "validation_audit.json",\n                "decoder_vocabulary.json", "mclip_text_features.pt", "mclip_prompt_bank.pt")\n\n\ndef require(condition, message):\n    if not condition:\n        raise ValueError(message)\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef load_checkpoint(path):\n    require(Path(path).is_file(), f"Selected checkpoint does not exist: {path}")\n    # No unsafe unpickling fallback. Legacy project artifacts load with weights_only=True.\n    return torch.load(path, map_location="cpu", weights_only=True)\n\n\nclass CheckpointReportDecoder(nn.Module):\n    def __init__(self, architecture=None):\n        super().__init__()\n        self.architecture = dict(ARCHITECTURE if architecture is None else architecture)\n        a = self.architecture\n        require(not a["scale_token_embeddings"] and not a["final_layer_norm"],\n                "This class implements the recorded Kaggle decoder only")\n        self.token_embedding = nn.Embedding(a["vocab_size"], a["d_model"], padding_idx=a["pad_id"])\n        self.position_embedding = nn.Embedding(a["max_tokens"], a["d_model"])\n        layer = nn.TransformerDecoderLayer(\n            d_model=a["d_model"], nhead=a["num_heads"], dim_feedforward=a["ffn_dim"],\n            dropout=a["dropout"], activation=a["activation"], batch_first=True,\n            norm_first=a["norm_first"])\n        self.transformer = nn.TransformerDecoder(layer, num_layers=a["num_layers"])\n        self.output_projection = nn.Linear(a["d_model"], a["vocab_size"])\n\n    def forward(self, condition, input_ids):\n        a = self.architecture\n        require(input_ids.ndim == 2 and 0 < input_ids.shape[1] <= a["max_tokens"],\n                "Decoder IDs must be [B,T] within the saved positional limit")\n        require(tuple(condition.shape) == (input_ids.shape[0], a["d_model"]),\n                "Condition dimensions incompatible with checkpoint decoder")\n        positions = torch.arange(input_ids.shape[1], device=input_ids.device).unsqueeze(0)\n        target = self.token_embedding(input_ids) + self.position_embedding(positions)\n        mask = torch.triu(torch.ones(input_ids.shape[1], input_ids.shape[1],\n                                    dtype=torch.bool, device=input_ids.device), diagonal=1)\n        hidden = self.transformer(target, condition.unsqueeze(1).to(target.dtype),\n                                  tgt_mask=mask, tgt_key_padding_mask=input_ids.eq(a["pad_id"]))\n        return self.output_projection(hidden)\n\n    @torch.no_grad()\n    def generate(self, condition, max_length=None):\n        self.eval()\n        a = self.architecture\n        limit = a["max_tokens"] if max_length is None else int(max_length)\n        require(2 <= limit <= a["max_tokens"], "Invalid generation length")\n        ids = torch.full((len(condition), 1), a["bos_id"], dtype=torch.long, device=condition.device)\n        finished = torch.zeros(len(condition), dtype=torch.bool, device=condition.device)\n        for _ in range(limit - 1):\n            next_id = self(condition, ids)[:, -1].argmax(-1)\n            next_id = torch.where(finished, torch.full_like(next_id, a["pad_id"]), next_id)\n            ids = torch.cat((ids, next_id[:, None]), dim=1)\n            finished |= next_id.eq(a["eos_id"])\n            if bool(finished.all()):\n                break\n        return ids\n\n\ndef validate_model_state(checkpoint, model):\n    require(isinstance(checkpoint, dict), "Checkpoint must be a state dictionary payload")\n    state = checkpoint.get("decoder")\n    require(isinstance(state, dict), "Required decoder state dictionary absent")\n    expected = model.state_dict()\n    require(set(state) == set(expected),\n            f"Decoder key mismatch: missing={sorted(set(expected)-set(state))}; unexpected={sorted(set(state)-set(expected))}")\n    for key, tensor in state.items():\n        require(isinstance(tensor, torch.Tensor) and tensor.shape == expected[key].shape,\n                f"Decoder shape mismatch: {key}")\n        require(tensor.dtype == expected[key].dtype, f"Decoder dtype mismatch: {key}")\n        require(bool(torch.isfinite(tensor).all()), f"Non-finite decoder weights: {key}")\n    if "architecture" in checkpoint:\n        require(checkpoint["architecture"] == model.architecture, "Saved architecture metadata differs")\n    for key, expected_value in (("vocabulary_size", model.architecture["vocab_size"]),\n                                ("max_tokens", model.architecture["max_tokens"])):\n        if key in checkpoint:\n            require(checkpoint[key] == expected_value, f"Saved {key} differs")\n\n\ndef inspect_training_state(checkpoint, model, best_checkpoint=None):\n    require(isinstance(checkpoint, dict), "Training checkpoint must be a dictionary payload")\n    required = {"epoch", "decoder", "optimizer", "history", "best_validation_bleu4", "epochs_without_improvement"}\n    require(required <= checkpoint.keys(), f"Required training state absent: {sorted(required-checkpoint.keys())}")\n    epoch = checkpoint["epoch"]\n    require(type(epoch) is int and epoch > 0, "Saved epoch must be a positive integer")\n    require(checkpoint.get("scheduler") is None or checkpoint.get("scheduler") == {},\n            "Unknown scheduler state; supply its verified implementation before resuming")\n    if "training_config" in checkpoint:\n        require(checkpoint["training_config"] == TRAINING, "Training configuration differs from recorded run")\n    history = checkpoint["history"]\n    require(isinstance(history, list) and len(history) == epoch, "History length differs from saved epoch")\n    require(all(isinstance(row, dict) and {"epoch", "training_loss", "validation_bleu4"} <= row.keys()\n                for row in history), "History record fields absent or invalid")\n    require([row["epoch"] for row in history] == list(range(1, epoch + 1)), "History is not contiguous")\n    for row in history:\n        require(all(math.isfinite(float(row[key])) for key in ("training_loss", "validation_bleu4")),\n                "Non-finite history value")\n    selected = max(history, key=lambda row: row["validation_bleu4"])\n    require(math.isclose(float(selected["validation_bleu4"]), float(checkpoint["best_validation_bleu4"]),\n                         rel_tol=0, abs_tol=1e-10), "History and saved best metric disagree")\n    require(type(checkpoint["epochs_without_improvement"]) is int and\n            checkpoint["epochs_without_improvement"] == epoch - selected["epoch"], "Patience state disagrees with history")\n    validate_model_state(checkpoint, model)\n    if best_checkpoint is not None:\n        validate_selected_best(checkpoint, model, best_checkpoint)\n    opt = checkpoint["optimizer"]\n    require(isinstance(opt, dict) and {"state", "param_groups"} <= opt.keys(), "Invalid optimizer state")\n    require(len(opt["param_groups"]) == 1, "Expected one decoder SGD parameter group")\n    group = opt["param_groups"][0]\n    for key, expected in dict(lr=0.001, momentum=0.99, dampening=0, weight_decay=0,\n                              nesterov=False, maximize=False).items():\n        require(group.get(key, False if key == "maximize" else None) == expected,\n                f"SGD setting mismatch: {key}")\n    parameters = list(model.named_parameters())\n    saved_ids = group["params"]\n    require(len(saved_ids) == len(parameters) and len(set(saved_ids)) == len(saved_ids),\n            "Optimizer parameter count or mapping incompatible")\n    require(set(opt["state"]) == set(saved_ids), "Missing or unexpected optimizer parameter states")\n    nonzero = 0\n    for pid, (name, parameter) in zip(saved_ids, parameters):\n        require(set(opt["state"][pid]) == {"momentum_buffer"}, f"Unknown/missing SGD state: {name}")\n        buffer = opt["state"][pid]["momentum_buffer"]\n        require(isinstance(buffer, torch.Tensor) and buffer.shape == parameter.shape and\n                buffer.dtype == parameter.dtype, f"Momentum shape/dtype incompatible: {name}")\n        require(bool(torch.isfinite(buffer).all()), f"Non-finite momentum: {name}")\n        nonzero += int(bool(buffer.ne(0).any()))\n    require(nonzero > 0, "All SGD momentum is zero")\n    if "parameter_names" in checkpoint:\n        require(checkpoint["parameter_names"] == [name for name, _ in parameters], "Optimizer parameter order differs")\n    return dict(epoch=epoch, next_epoch=epoch + 1, best_epoch=selected["epoch"],\n                best_validation_bleu4=float(selected["validation_bleu4"]),\n                epochs_without_improvement=checkpoint["epochs_without_improvement"],\n                momentum_buffers=len(saved_ids), nonzero_momentum_buffers=nonzero,\n                parameter_count=sum(p.numel() for _, p in parameters),\n                legacy_rng_missing="rng_state" not in checkpoint,\n                scheduler="none", compatibility="PASS")\n\n\ndef validate_selected_best(checkpoint, model, best_checkpoint):\n    require(isinstance(best_checkpoint, dict), "Selected best must be a dictionary payload")\n    selected = max(checkpoint["history"], key=lambda row: row["validation_bleu4"])\n    require(best_checkpoint.get("epoch") == selected["epoch"] and\n            math.isclose(float(best_checkpoint.get("validation_bleu4", float("nan"))),\n                         float(selected["validation_bleu4"]), rel_tol=0, abs_tol=1e-10),\n            f"Best checkpoint disagrees with history: need epoch {selected[\'epoch\']}, "\n            f"found epoch {best_checkpoint.get(\'epoch\')}. Recover matching weights; do not substitute latest weights.")\n    validate_model_state(best_checkpoint, model)\n\n\ndef restore_weights_optimizer(checkpoint, model, device="cpu"):\n    """Restore latest state for diagnostics; not authorization to continue training."""\n    result = inspect_training_state(checkpoint, model)\n    model.load_state_dict(checkpoint["decoder"], strict=True)\n    model.to(device)\n    optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.99)\n    optimizer.load_state_dict(checkpoint["optimizer"])\n    for parameter, state in optimizer.state.items():\n        state["momentum_buffer"] = state["momentum_buffer"].to(parameter.device)\n    for pid, parameter in zip(checkpoint["optimizer"]["param_groups"][0]["params"], model.parameters()):\n        require(torch.equal(optimizer.state[parameter]["momentum_buffer"].cpu(),\n                            checkpoint["optimizer"]["state"][pid]["momentum_buffer"].cpu()),\n                "Momentum changed during restoration")\n    model.eval()\n    return optimizer, result\n\n\ndef restore_training(checkpoint, model, best_checkpoint, device="cpu"):\n    result = inspect_training_state(checkpoint, model, best_checkpoint)\n    optimizer, _ = restore_weights_optimizer(checkpoint, model, device)\n    return optimizer, result\n\n\ndef read_manifest(directory, name):\n    with (Path(directory) / name).open(encoding="utf-8-sig", newline="") as handle:\n        return list(csv.DictReader(handle))\n\n\ndef validate_artifacts(directory, expected_hashes=None):\n    directory = Path(directory)\n    hashes = {name: sha256(directory / name) for name in DEPENDENCIES}\n    if expected_hashes is not None:\n        require(hashes == expected_hashes, "Feature/vocabulary/split dependency hashes differ")\n    records = {name: read_manifest(directory, name + "_manifest.csv")\n               for name in ("train", "validation", "test", "prompt", "validated")}\n    ids = {}\n    for name, count in dict(train=571, validation=71, test=72, prompt=500, validated=714).items():\n        rows = records[name]\n        ids[name] = [row["image_id"] for row in rows]\n        require(len(rows) == count and len(set(ids[name])) == count, f"{name} count/ID uniqueness failed")\n    train, val, test = (set(ids[name]) for name in ("train", "validation", "test"))\n    require(not (train & val or train & test or val & test), "Split leakage")\n    require(train | val | test == set(ids["validated"]), "Split population differs")\n    require(set(ids["prompt"]) <= train, "Prompt bank contains non-training IDs")\n    by_id = {r["image_id"]: r for r in records["validated"]}\n    for name in ("train", "validation", "test", "prompt"):\n        require(all(row["report"] == by_id[row["image_id"]]["report"] for row in records[name]),\n                f"{name} reports differ from validated manifest")\n    audit = json.loads((directory / "validation_audit.json").read_text(encoding="utf-8"))\n    require(audit["csv_rows"] == 726 and audit["physical_images"] == 746, "Raw audit population differs")\n    vocabulary = json.loads((directory / "decoder_vocabulary.json").read_text(encoding="utf-8"))\n    require(vocabulary["max_tokens"] == 160 and len(vocabulary["vocabulary"]) == 368,\n            "Vocabulary or positional capacity differs")\n    require(vocabulary["special_token_ids"] == dict(pad=0, bos=1, eos=2, unk=3), "Special-token IDs differ")\n    expected_vocab = ["<pad>", "<bos>", "<eos>", "<unk>"] + sorted({\n        token for row in records["train"] for token in re.findall(r"\\w+|[^\\w\\s]", row["report"])})\n    require(vocabulary["vocabulary"] == expected_vocab, "Vocabulary is not the saved training-only lexicographic mapping")\n    text = load_checkpoint(directory / "mclip_text_features.pt")\n    require(text["model"] == MCLIP_NAME and text["normalization"] == "l2", "Text encoder metadata differs")\n    for name in ("train", "validation"):\n        require(set(text[name]) == set(ids[name]), f"{name} feature membership differs")\n        for vector in text[name].values():\n            require(vector.shape == (768,) and bool(torch.isfinite(vector).all()) and\n                    torch.allclose(vector.norm(), torch.tensor(1.0), atol=1e-4), "Invalid normalized text feature")\n    prompt = load_checkpoint(directory / "mclip_prompt_bank.pt")\n    require(prompt["model"] == MCLIP_NAME and prompt.get("normalization") == "raw", "Prompt encoder/normalization differs")\n    require(prompt["prompt_ids"] == ids["prompt"], "Prompt tensor row order/provenance differs")\n    vectors = prompt["embeddings"]\n    require(vectors.shape == (500, 768) and bool(torch.isfinite(vectors).all()) and\n            bool((vectors.norm(dim=-1) > 0).all()), "Invalid prompt bank")\n    aligned = torch.stack([text["train"][key] for key in ids["prompt"]])\n    # Separate encodings are not bitwise equal. Record the observed difference;\n    # membership and frozen-cache hashes above are the provenance assertions.\n    normalized_prompts = F.normalize(vectors, dim=-1)\n    cosine = (normalized_prompts * aligned).sum(-1)\n    cache_comparison = dict(max_absolute_difference=float((normalized_prompts - aligned).abs().max()),\n                            minimum_cosine=float(cosine.min()), mean_cosine=float(cosine.mean()),\n                            exact_encoder_revision="UNVERIFIED")\n    return dict(records=records, vocabulary=vocabulary, text=text, prompt=prompt,\n                dependency_hashes=hashes, prompt_text_comparison=cache_comparison)\n\n\ndef encode_report(text, vocabulary):\n    lookup = {token: i for i, token in enumerate(vocabulary)}\n    ids = [1] + [lookup.get(token, 3) for token in re.findall(r"\\w+|[^\\w\\s]", text)] + [2]\n    require(len(ids) <= 160, "Report exceeds saved positional capacity; no silent truncation")\n    return ids\n\n\ndef decode_report(token_ids, vocabulary):\n    tokens = []\n    for raw in token_ids:\n        token = int(raw)\n        if token == 2:\n            break\n        require(0 <= token < len(vocabulary), "Generated token outside vocabulary")\n        if token not in {0, 1}:\n            tokens.append(vocabulary[token])\n    text = re.sub(r"\\s+([.,!?;:%)\\]])", r"\\1", " ".join(tokens))\n    return re.sub(r"([(\\[])\\s+", r"\\1", text).strip()\n\n\ndef enrich(image, prompts):\n    require(image.ndim == prompts.ndim == 2 and image.shape[-1] == prompts.shape[-1] == 768,\n            "Image/prompt dimensions must share the recorded 768-D CLIP space")\n    require(bool(torch.isfinite(image).all()) and bool(torch.isfinite(prompts).all()), "Non-finite encoder output")\n    scores = image @ prompts.T / math.sqrt(768)\n    attention = scores.softmax(-1)\n    residual = image + attention @ prompts\n    require(bool((residual.norm(dim=-1) > 0).all()), "Zero residual embedding")\n    phi = F.normalize(residual, p=2, dim=-1)\n    require(bool(torch.isfinite(phi).all()), "Non-finite enriched features")\n    return phi, attention\n\n\ndef capture_rng():\n    return dict(python=random.getstate(), torch_cpu=torch.get_rng_state(),\n                torch_cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [])\n\n\ndef restore_rng(state):\n    random.setstate(state["python"])\n    torch.set_rng_state(state["torch_cpu"].cpu())\n    if state["torch_cuda"]:\n        require(len(state["torch_cuda"]) == torch.cuda.device_count(), "CUDA RNG device count differs")\n        torch.cuda.set_rng_state_all([value.cpu() for value in state["torch_cuda"]])\n\n\ndef atomic_save(payload, destination):\n    destination = Path(destination)\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    temporary = destination.with_suffix(destination.suffix + ".tmp")\n    with temporary.open("wb") as handle:\n        torch.save(payload, handle)\n        handle.flush()\n        os.fsync(handle.fileno())\n    os.replace(temporary, destination)\n\n\ndef training_payload(model, optimizer, epoch, history, best_metric, bad_epochs, hashes, provenance):\n    return dict(schema_version="zeromrg_covctr_gate1_v1", epoch=epoch, decoder=model.state_dict(),\n                optimizer=optimizer.state_dict(), scheduler=None, history=history,\n                best_validation_bleu4=best_metric, epochs_without_improvement=bad_epochs,\n                architecture=model.architecture, parameter_names=[n for n, _ in model.named_parameters()],\n                training_config=TRAINING, dependency_hashes=hashes, provenance=provenance,\n                rng_state=capture_rng())\n'
RECOVERY_RUNTIME_SOURCE = '"""Separate recovery selection with historical stopping; no execution on import."""\nfrom __future__ import annotations\n\nimport copy\nimport hashlib\nimport json\nimport math\nimport random\nfrom pathlib import Path\n\nimport torch\n\nfrom zeromrg_gate1_runtime import (\n    ARCHITECTURE, TRAINING, atomic_save, capture_rng, inspect_training_state,\n    load_checkpoint, require, restore_rng, restore_weights_optimizer, sha256,\n    validate_model_state,\n)\n\nORIGINAL_FILES = {\n    "text_decoder_last.pt": "ecc204f031b0fa83b4734c3228f20ba2172ce291288f31a6551921126546c299",\n    "text_decoder_best.pt": "dba64646e89f1c55c69c9e3a7361141829e4dc7ec32114f98f2edce8b89a0568",\n    "text_training_history.json": "ccd65f2ec0ce2f569f683233f5a13c48c26bb27075aa9661bc7a272c3a14a015",\n}\nORIGINAL_RECORD = dict(epoch=90, best_epoch=85, best_validation_bleu4=56.44423922571333,\n                       epochs_without_improvement=5, patience=50,\n                       selected_weights_available=False, stale_best_epoch=8,\n                       files=ORIGINAL_FILES)\nRECOVERY_POLICY = dict(id="A_historical_stop_available_best_v1", baseline_epoch=90,\n                       selection="strict_validation_BLEU4_improvement_in_epochs_90_onward",\n                       stopping="original_history_plus_continuation_patience_50",\n                       max_epoch=200, max_epochs_per_session=5,\n                       historical_weights_missing=True,\n                       deviation="selection_restricted_to_recoverable_window; legacy_RNG_lost")\nRECOVERY_SCHEMA = "zeromrg_gate3_dual_ledger_v1"\n\n\ndef plain_metadata(value):\n    """Normalize str subclasses (including TorchVersion) into safe plain types."""\n    return json.loads(json.dumps(value, allow_nan=False))\n\n\ndef json_sha256(value):\n    raw = json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()\n    return hashlib.sha256(raw).hexdigest()\n\n\ndef capture_recovery_rng():\n    import numpy as np\n    result = capture_rng()\n    name, keys, position, has_gauss, cached = np.random.get_state()\n    result["numpy"] = dict(name=name, keys=keys.tolist(), position=int(position),\n                            has_gauss=int(has_gauss), cached=float(cached))\n    return result\n\n\ndef restore_recovery_rng(state):\n    import numpy as np\n    require(isinstance(state, dict) and {"python", "torch_cpu", "torch_cuda", "numpy"} <= state.keys(),\n            "Required recovery RNG state absent")\n    restore_rng(state)\n    value = state["numpy"]\n    np.random.set_state((value["name"], np.asarray(value["keys"], dtype=np.uint32),\n                         value["position"], value["has_gauss"], value["cached"]))\n\n\ndef seed_legacy_recovery():\n    import numpy as np\n    random.seed(42)\n    np.random.seed(42)\n    torch.manual_seed(42)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(42)\n\n\ndef historical_stop_record(original_history, continuation_history):\n    history = original_history + continuation_history\n    selected = max(history, key=lambda row: row["validation_bleu4"])\n    return dict(best_epoch=selected["epoch"], best_validation_bleu4=float(selected["validation_bleu4"]),\n                epochs_without_improvement=history[-1]["epoch"] - selected["epoch"], patience=50)\n\n\ndef safe_name(name):\n    require(isinstance(name, str) and name and Path(name).name == name and\n            name not in {".", ".."} and "/" not in name and "\\\\" not in name,\n            "Artifact reference must be a relative basename")\n    return name\n\n\ndef valid_digest(value):\n    return isinstance(value, str) and len(value) == 64 and all(c in "0123456789abcdef" for c in value)\n\n\ndef validate_measurement(row):\n    required = {"epoch", "stage", "validation_bleu4", "predictions_file", "predictions_sha256",\n                "metric_fingerprint_sha256"}\n    require(isinstance(row, dict) and required <= row.keys(), "Validation measurement fields absent")\n    require(type(row["epoch"]) is int and row["epoch"] >= 90, "Invalid measured epoch")\n    require(math.isfinite(float(row["validation_bleu4"])) and 0 <= float(row["validation_bleu4"]) <= 100,\n            "Invalid measured BLEU4")\n    safe_name(row["predictions_file"])\n    require(valid_digest(row["predictions_sha256"]) and valid_digest(row["metric_fingerprint_sha256"]),\n            "Invalid measurement integrity digest")\n\n\ndef new_recovery_metadata(original_history, baseline, recovery_id, hashes, metric_fingerprint, provenance):\n    validate_measurement(baseline)\n    require(baseline["epoch"] == 90 and baseline["stage"] == "baseline", "Must measure epoch90 baseline first")\n    require(len(original_history) == 90, "Original history must contain exactly90 epochs")\n    require(baseline["metric_fingerprint_sha256"] == json_sha256(metric_fingerprint), "Baseline metric identity differs")\n    return dict(schema_version=RECOVERY_SCHEMA, epoch=90, recovery_id=recovery_id,\n                recovery_policy=copy.deepcopy(RECOVERY_POLICY), historical_record=copy.deepcopy(ORIGINAL_RECORD),\n                original_history=copy.deepcopy(original_history), continuation_history=[],\n                recovery_history=[copy.deepcopy(baseline)],\n                original_stop_tracking=historical_stop_record(original_history, []),\n                dependency_hashes=copy.deepcopy(hashes), metric_fingerprint=plain_metadata(metric_fingerprint),\n                provenance=plain_metadata(provenance))\n\n\ndef append_recovery_epoch(metadata, epoch, training_loss, measurement):\n    """Pure metadata update; no optimizer/model operation and no historical mutation."""\n    validate_measurement(measurement)\n    require("decoder" not in metadata and "optimizer" not in metadata, "Pass metadata only, not model/optimizer tensors")\n    require(epoch == metadata["epoch"] + 1 and measurement["epoch"] == epoch and\n            measurement["stage"] == "continuation", "Continuation epoch is not the next completed epoch")\n    require(epoch <= 200 and metadata["original_stop_tracking"]["epochs_without_improvement"] < 50,\n            "Original stopping condition already met")\n    require(math.isfinite(float(training_loss)), "Non-finite training loss")\n    require(measurement["metric_fingerprint_sha256"] == json_sha256(metadata["metric_fingerprint"]),\n            "Metric implementation changed")\n    current_best = max(metadata["recovery_history"], key=lambda row: row["validation_bleu4"])\n    improved = float(measurement["validation_bleu4"]) > float(current_best["validation_bleu4"])\n    result = copy.deepcopy(metadata)\n    # This function accepts metadata only, never a payload containing model tensors.\n    result["epoch"] = epoch\n    result["continuation_history"].append(dict(epoch=epoch, training_loss=float(training_loss),\n                                               validation_bleu4=float(measurement["validation_bleu4"])))\n    result["recovery_history"].append(copy.deepcopy(measurement))\n    result["original_stop_tracking"] = historical_stop_record(result["original_history"], result["continuation_history"])\n    return result, improved\n\n\ndef recovery_best_payload(model, measurement, recovery_id, dependency_hashes):\n    validate_measurement(measurement)\n    return dict(schema_version="zeromrg_gate3_recovery_best_v1", decoder=model.state_dict(),\n                epoch=measurement["epoch"], validation_bleu4=float(measurement["validation_bleu4"]),\n                validation_predictions_sha256=measurement["predictions_sha256"],\n                metric_fingerprint_sha256=measurement["metric_fingerprint_sha256"],\n                recovery_id=recovery_id, selection_scope="recovery_epochs_90_onward",\n                historical_selected_epoch=85, is_missing_epoch85_model=False,\n                architecture=model.architecture, vocabulary_size=model.architecture["vocab_size"],\n                max_tokens=model.architecture["max_tokens"], dependency_hashes=dependency_hashes)\n\n\ndef exclusive_save(payload, path):\n    require(not Path(path).exists(), "Immutable artifact already exists; refuse overwrite")\n    atomic_save(payload, path)\n\n\ndef best_reference(path, measurement):\n    return dict(file=safe_name(Path(path).name), sha256=sha256(path), epoch=measurement["epoch"],\n                validation_bleu4=float(measurement["validation_bleu4"]),\n                predictions_file=safe_name(measurement["predictions_file"]),\n                predictions_sha256=measurement["predictions_sha256"])\n\n\ndef latest_recovery_payload(metadata, model, optimizer, reference):\n    result = copy.deepcopy(metadata)\n    result.update(decoder=model.state_dict(), optimizer=optimizer.state_dict(), scheduler=None,\n                  architecture=model.architecture, parameter_names=[n for n, _ in model.named_parameters()],\n                  training_config=TRAINING, recovery_best=copy.deepcopy(reference), rng_state=capture_recovery_rng())\n    return result\n\n\ndef metadata_from_payload(payload):\n    return {key: copy.deepcopy(payload[key]) for key in\n            ("schema_version", "epoch", "recovery_id", "recovery_policy", "historical_record", "original_history",\n             "continuation_history", "recovery_history", "original_stop_tracking", "dependency_hashes",\n             "metric_fingerprint", "provenance")}\n\n\ndef legacy_view(payload):\n    stop = payload["original_stop_tracking"]\n    return dict(epoch=payload["epoch"], decoder=payload["decoder"], optimizer=payload["optimizer"],\n                history=payload["original_history"] + payload["continuation_history"],\n                best_validation_bleu4=stop["best_validation_bleu4"],\n                epochs_without_improvement=stop["epochs_without_improvement"],\n                scheduler=payload["scheduler"], architecture=payload["architecture"],\n                parameter_names=payload["parameter_names"], training_config=payload["training_config"],\n                rng_state=payload["rng_state"])\n\n\ndef validate_recovery_state(payload, model, original_history, hashes, best=None):\n    require(isinstance(payload, dict), "Recovery checkpoint must be a dictionary")\n    required = {"schema_version", "epoch", "recovery_id", "recovery_policy", "historical_record", "original_history",\n                "continuation_history", "recovery_history", "original_stop_tracking", "dependency_hashes",\n                "metric_fingerprint", "provenance", "decoder", "optimizer", "scheduler", "architecture",\n                "parameter_names", "training_config", "recovery_best", "rng_state"}\n    require(required <= payload.keys(), "Required recovery state absent: " + str(sorted(required - payload.keys())))\n    require(payload["schema_version"] == RECOVERY_SCHEMA and payload["recovery_policy"] == RECOVERY_POLICY,\n            "Unknown or altered recovery policy")\n    require(payload["historical_record"] == ORIGINAL_RECORD and payload["original_history"] == original_history,\n            "Original historical evidence changed")\n    require(payload["dependency_hashes"] == hashes, "Split/feature/vocabulary dependencies changed")\n    require(payload["scheduler"] is None, "This recovery protocol has no scheduler")\n    epoch = payload["epoch"]\n    require(type(epoch) is int and 90 <= epoch <= 200, "Recovery epoch outside approved protocol")\n    continuation = payload["continuation_history"]\n    require(isinstance(continuation, list) and all(isinstance(row, dict) and\n            {"epoch", "training_loss", "validation_bleu4"} <= row.keys() for row in continuation),\n            "Continuation history fields absent")\n    require(len(original_history) == 90 and len(continuation) == epoch - 90, "Historical/continuation lengths disagree")\n    require([row["epoch"] for row in continuation] == list(range(91, epoch + 1)), "Continuation history is not contiguous")\n    tracking = historical_stop_record(original_history, continuation)\n    require(payload["original_stop_tracking"] == tracking, "Original stopping ledger changed/reset")\n    for index in range(len(continuation)):\n        require(historical_stop_record(original_history, continuation[:index])["epochs_without_improvement"] < 50,\n                "Continuation advanced beyond original early stopping")\n    rows = payload["recovery_history"]\n    require(len(rows) == epoch - 89 and [r["epoch"] for r in rows] == list(range(90, epoch + 1)),\n            "Recovery selection history is not contiguous")\n    for index, row in enumerate(rows):\n        validate_measurement(row)\n        require(row["stage"] == ("baseline" if index == 0 else "continuation"), "Recovery measurement stage differs")\n        require(row["metric_fingerprint_sha256"] == json_sha256(payload["metric_fingerprint"]),\n                "Validation metric fingerprints disagree")\n        if index:\n            require(float(row["validation_bleu4"]) == float(continuation[index - 1]["validation_bleu4"]),\n                    "Validation score differs between ledgers")\n    selected = max(rows, key=lambda row: row["validation_bleu4"])\n    reference = payload["recovery_best"]\n    require(isinstance(reference, dict) and {"file", "sha256", "epoch", "validation_bleu4", "predictions_file",\n                                          "predictions_sha256"} <= reference.keys(), "Recovery best reference fields absent")\n    safe_name(reference["file"])\n    require(valid_digest(reference["sha256"]), "Invalid selected-weight checksum")\n    require(reference["epoch"] == selected["epoch"] and reference["validation_bleu4"] == selected["validation_bleu4"] and\n            reference["predictions_file"] == selected["predictions_file"] and\n            reference["predictions_sha256"] == selected["predictions_sha256"], "Recovery best does not match measured validation history")\n    require(isinstance(payload["rng_state"], dict) and\n            {"python", "torch_cpu", "torch_cuda", "numpy"} <= payload["rng_state"].keys(), "Recovery RNG state absent")\n    result = inspect_training_state(legacy_view(payload), model)\n    if best is not None:\n        require(best.get("schema_version") == "zeromrg_gate3_recovery_best_v1" and\n                best.get("recovery_id") == payload["recovery_id"] and best.get("epoch") == selected["epoch"] and\n                best.get("validation_bleu4") == selected["validation_bleu4"] and\n                best.get("validation_predictions_sha256") == selected["predictions_sha256"] and\n                best.get("metric_fingerprint_sha256") == selected["metric_fingerprint_sha256"] and\n                best.get("dependency_hashes") == hashes and best.get("selection_scope") == "recovery_epochs_90_onward" and\n                best.get("is_missing_epoch85_model") is False and best.get("historical_selected_epoch") == 85,\n                "Selected recovery weights have invalid provenance/score")\n        validate_model_state(best, model)\n    result.update(recovery_best_epoch=selected["epoch"], recovery_best_validation_bleu4=selected["validation_bleu4"],\n                  recovery_non_improvement_counter=epoch - selected["epoch"],\n                  historical_rng_missing=True,\n                  stop=tracking["epochs_without_improvement"] >= 50 or epoch >= 200)\n    return result\n\n\ndef validate_recovery_files(payload, model, original_history, hashes, directory, validation_ids):\n    directory = Path(directory)\n    ref = payload["recovery_best"]\n    best_path = directory / safe_name(ref["file"])\n    require(sha256(best_path) == ref["sha256"], "Selected recovery weights checksum mismatch")\n    best = load_checkpoint(best_path)\n    result = validate_recovery_state(payload, model, original_history, hashes, best)\n    for row in payload["recovery_history"]:\n        path = directory / safe_name(row["predictions_file"])\n        require(sha256(path) == row["predictions_sha256"], "Validation predictions checksum mismatch")\n        prediction = json.loads(path.read_text(encoding="utf-8"))\n        require(isinstance(prediction, dict) and {"split", "image_ids", "epoch", "validation_bleu4",\n                "metric_fingerprint_sha256", "references", "predictions"} <= prediction.keys(),\n                "Validation prediction fields absent")\n        require(prediction["split"] == "validation" and prediction["image_ids"] == validation_ids,\n                "Validation predictions contain wrong IDs/split; possible leakage")\n        require(len(prediction["references"]) == len(validation_ids) == len(prediction["predictions"]) and\n                all(isinstance(text, str) for text in prediction["references"] + prediction["predictions"]),\n                "Validation references/predictions length or types differ")\n        require(prediction["epoch"] == row["epoch"] and prediction["validation_bleu4"] == row["validation_bleu4"] and\n                prediction["metric_fingerprint_sha256"] == row["metric_fingerprint_sha256"], "Saved validation measurement differs")\n    return result\n\n\ndef restore_recovery(payload, model, original_history, hashes, directory, validation_ids, device="cpu"):\n    result = validate_recovery_files(payload, model, original_history, hashes, directory, validation_ids)\n    optimizer, _ = restore_weights_optimizer(legacy_view(payload), model, device)\n    # RNG restored again immediately before training, after any diagnostics/evaluation.\n    restore_recovery_rng(payload["rng_state"])\n    return optimizer, result\n'
RUNTIME_HASHES = {'gate1': 'e897aa5e800ce18911744cc13e16b75b239477a4edb8bc1df0f8e9ab3f9e038a', 'gate3': '1fbca913016996971e369a909397293b74284e15d582d51b6376b95b7f2bbe5a'}
METRIC_LOCK = 'pycocoevalcap==1.2 --hash=sha256:083ed7910f1aec000b0a237ef6665f74edf19954204d0b1cbdb8399ed132228d\n'
METRIC_WHEEL_SHA256 = '083ed7910f1aec000b0a237ef6665f74edf19954204d0b1cbdb8399ed132228d'
METRIC_IMPLEMENTATION = {'pycocoevalcap/tokenizer/ptbtokenizer.py': 'ded90551127b8a4bf683873432ccafa83330a9997f6d0b695cb71a618808b8c1', 'pycocoevalcap/tokenizer/stanford-corenlp-3.4.1.jar': '2fcb91bb7a111f93d71e264f4ee0e3afd19ba0dde6d21b38605088df9e940399', 'pycocoevalcap/bleu/bleu.py': '05b74a92b32e071eacfb954ef34328137cfd1b82c563572648633087def23776', 'pycocoevalcap/bleu/bleu_scorer.py': '413ccfef71b88c5f11d25d05fdcf08625a1559338ffbb7c2787e800116eaecdd'}
HISTORICAL_REPORTS = {'zeromrg_gate1_readiness.md': "# ZeroMRG Gate 1 readiness decision\n\n> **Subsequent Gate 2 evidence, 2026-10-11 IST:** this remains the historical pre-execution Gate 1 assessment. One bounded private preflight was explicitly approved and executed afterward. Actual Kaggle mounts/latest restoration/finite forward/disposable save-restore passed; the expected best85-versus-file8 guard stopped execution. See [Gate 2 record](zeromrg_gate2_preflight.md) for current results. Selected-best readiness is still BLOCKED; no training or further remote execution is authorized.\n\nDate: 2026-10-10. Repository `E:\\Insha Project\\ZeroMRG`, account `inshasadaf114`; base ZeroMRG/COV-CTR **0%-paired only**. Gate 0 evidence was verified against current source/artifacts rather than accepted as proof. **Gate 1: BLOCKED**, with local preparation and latest-state compatibility complete. No Kaggle job, GPU training, final test evaluation, remote publication, Git commit or push was performed.\n\n## Acceptance record\n\n| Requirement | Status | Evidence / limitation |\n|---|---|---|\n| Canonical source identified | PASS | Historical checkpoint-producing decoder and cached dependencies, embedded self-contained Gate 1 runtime; original notebooks retained |\n| Canonical latest recovery verified | PASS, local | Epoch90 full decoder/SGD/history, archive CRC and SHA-256; remote latest was not independently downloaded/hash-verified |\n| Selected-best recovery | **BLOCKED** | History85/56.44423922571333 versus actual best8/52.33887482243904; independent remote best has identical stale bytes |\n| Checkpoint-driven continuation | PASS, prepared | Explicit selected path/schema; derives next epoch91, strict restore; refuses missing/incompatible best and no scratch fallback |\n| Model and optimizer compatibility | PASS, CPU | Actual 57,399,152-parameter architecture, 112 state tensors, 112/112 finite nonzero SGD momentum buffers, exact restoration |\n| Dataset/prompt assertions | PASS, implemented; mounted execution UNVERIFIED | Saved714 pairs, 571/71/72, train-only500 prompts, vocabulary368, features768 and pinned hashes; raw726/746 assertions on Kaggle prepared |\n| Zero-shot paper inspection | PASS, inspection | Named companion image encoder and Eqs5/6 checked; historical encoder revision UNVERIFIED; no image-path execution evidence |\n| Kaggle preflight notebook/metadata | PASS, prepared | New private slug, exact dataset references, disabled training; cell compilation/source/hash/metadata checks passed |\n| GPU preflight execution | UNVERIFIED | Not approved or run; current best causes explicit BLOCKED report after latest diagnostics if earlier checks pass |\n| Persistence | PASS, design/code; new durability UNVERIFIED | Latest atomic every epoch, actual-best bundles, max-five-epoch chunks; future publish/retrieval requires approval/evidence |\n| Git review/documentation | PASS, review | User work retained, controlled commit plan below; no staging, commit, discard or push |\n\nThe missing selected-best artifact is material to complete recovery and faithful downstream selection. This record therefore does not call the overall gate PASS or CONDITIONAL PASS. A technical preflight PASS, if eventually obtained, would describe that diagnostic only and would not prove historical revisions, convergence or final scientific results.\n\n## Delivered local artifacts\n\n- [Source reconciliation](zeromrg_source_reconciliation.md), [checkpoint recovery](zeromrg_checkpoint_recovery.md), this readiness record and [persistence procedure](zeromrg_checkpoint_persistence.md).\n- [Preflight notebook](../notebooks/gate1/preflight/ZeroMRG_COV_CTR_Preflight_Only_Kaggle.ipynb) and adjacent `kernel-metadata.json`.\n- [Checkpoint-driven continuation notebook](../notebooks/gate1/continuation/ZeroMRG_COV_CTR_Checkpoint_Continuation_Kaggle.ipynb) and adjacent `kernel-metadata.json`.\n- Canonical runtime, notebook builder, archive audit and static validator under `scripts/`; restoration/architecture tests in `tests/test_gate1_recovery.py`; evidence under `docs/evidence/`.\n- Corrected current [progress record](kaggle_covctr_progress.md), superseded [epoch95 guide](kaggle_epoch95_resume_guide.md), and Gate 0 correction. The old guide snapshot is retained in the ignored reconciliation directory; original notebooks are unchanged.\n\n## Verification performed\n\nThe two notebooks are valid JSON with unique cell IDs and cleared outputs. All Python cells compile; embedded runtime and pinned source/dependency hashes match the audited files. Metadata uses private new kernels, GPU enabled/TPU disabled, and the original two dataset references. Preflight contains no backward or optimizer step. Continuation has `ENABLE_APPROVED_TRAINING=False`, checks selected-best restoration before the training cell and restricts enabled training to at most five epochs per chunk. Source identifiers and final notebook hashes are recorded in [static validation evidence](evidence/gate1_notebook_validation.json).\n\n**51 lightweight CPU tests passed**, including 12 Gate 1 tests covering missing/corrupt state, full momentum restoration, next-epoch derivation, stale best rejection, unknown scheduler rejection, architecture/dtype/finite tensor checks, unscaled embeddings/no final norm, causal decoding, EOS/length, raw prompt equations, disposable full-state restore and future RNG restoration. The production parameter count was inspected on the meta device. Other repository tests use fixtures/synthetic data; running existing IU-Xray parser tests did not start an IU-Xray experiment. Actual archived decoder and optimizer restoration was checked on CPU independently without a real-model forward or optimizer step. [Test record](evidence/gate1_local_test_results.json).\n\nSource/import readiness is static: the notebooks embed their own decoder and utilities, use Torch/Python standard library for latest-state diagnostics, and do not require `src` checkout or missing historical text-model globals. Image preflight subsequently needs official `clip`/Pillow; continuation validation needs `pycocoevalcap` and Java. Availability/download/runtime compatibility of those packages on the chosen Kaggle image is not verified locally. No blanket dependency upgrade is prescribed. Exact mount paths can differ; the user must set the selected paths and assertions must pass. Metadata attachment names do not prove mounted contents.\n\n## Recovery and scientific decisions\n\n**Epoch91 SGD:** the local epoch90 state is complete enough for the next update, preserving momentum, best metric and counter. The prepared default continuation is nevertheless blocked because it cannot recover the historical winner at early stopping if no future improvement occurs. Missing best does not erase early-stopping counters; never reset them. A separately approved incomplete-selection policy is described in the recovery document but is not implemented/enabled. Historical RNG is absent, so exact trajectory replay is impossible even if epoch85 is recovered. New seeded recovery is an explicitly documented continuation, not bitwise replay.\n\n**Zero-shot:** the available epoch8 and epoch90 decoders are structurally capable of image conditioning. A transparently labelled diagnostic using a nonselected model could test mechanics, but would not satisfy this reproduction's validation-selected final inference protocol. The prepared image diagnostic requires a matching selected best. No valid selected-best 0%-paired final result can be claimed with current artifacts, and no held-out test evaluation has occurred. Do not relabel text-reconstruction BLEU as image-generation BLEU.\n\n**Training sequence:** diagnostic latest-state preflight first after approval, recover genuine85, then the one-training-image frozen path diagnostic, then explicitly approved decoder-only continuation preserving SGD/early-stopping conditions, then final held-out evaluation after defensible stopping/model selection. Epoch90 has not met patience50 and therefore does not establish that training is complete. The implementation's ceiling200 is not a paper-required target. Actual GPU costs are unmeasured; a single forward diagnostic is much cheaper than 45 non-improving epochs or as many as110 further epochs under the inherited ceiling. Test data never informs these choices.\n\nUnresolved: epoch85 weights; later persistent checkpoints; remote latest hash; original RNG; exact historical encoder/code/package revisions; fully self-contained historical text encoding source; reason for prompt/text cache differences; actual GPU imports/mounts/image forward; final selected-best test metrics; independently retrieved future persistence exports. Paper omissions for decoder/tokenizer/search/split settings remain explicitly documented assumptions. No 10%/100% paired or IU-Xray runs are authorized.\n\n## Git review and controlled commit plan\n\nBaseline HEAD is `fd2a370a9c3fd4fe427a4f94ae3ea480ed8e315f`. Before Gate 1, `.gitignore` and `tests/test_models.py` were modified; progress/CLI/Gate0/epoch95 docs, legacy resume notebook, evaluation modules/test, experiment driver, wrappers and diagnostic results were untracked. `.gitignore`'s `.kaggle/` exclusion is meaningful and retained. `tests/test_models.py` had only 11 added blank lines; verified nonblank content equality allows removal of only those accidental whitespace edits. All meaningful user changes and untracked artifacts are preserved. Git review evidence records final status/diff summaries; no broad cleanup was attempted.\n\nProposed controlled commits, after owner review: (1) source/audit/recovery/persistence/readiness docs plus small nonsecret evidence and targeted historical-document corrections; (2) canonical Gate 1 runtime/builder/static validator, two new notebooks/metadata and Gate 1 tests. Existing wrappers/CLI/evaluation/experiment-driver changes require separate review for ownership and scope; do not silently include them in Gate 1. The old resume notebook may be committed as historical material only if explicitly reviewed; it remains superseded. Inspect exact staged diff and `git diff --check` before committing. No commit or push is performed here.\n\nUse explicit file paths, never blanket `git add .`: exclude `.kaggle`, credential files, `venv`, `artifacts`, downloaded source outputs/archives, large weights/features, raw medical data and unrelated `results`. Existing CPU-smoke results are diagnostic, not scientific results. Preserve them locally. A commit plan does not authorize staging or remote push.\n\n## Exact next Kaggle action\n\n**No remote execution is currently approved.** The next proposed bounded action, requiring explicit owner approval, is to execute only the new private preflight kernel:\n\n```powershell\n.\\scripts\\kaggle.ps1 -KaggleArgs @('kernels','push','-p','notebooks/gate1/preflight','-t','1200')\n```\n\nThis creates/executes the new `inshasadaf114/zeromrg-covctr-gate1-preflight` version; it does not overwrite `zeromrg-workflow`. The 1200-second timeout is a bounded diagnostic budget, not a promise of completion. Current inputs should produce a BLOCKED selected-best report after successful latest checks and stop before CLIP; earlier mount/package/GPU failures are still possible. No training or test scoring is in this notebook. Do not push the continuation kernel or enable its training flag.\n\nAfter approval/execution, read status, retrieve that preflight's outputs into a new non-overwriting directory, inspect the report and preserve its source/output hashes. Never call the next step executed or passed until those artifacts are actually retrieved. Recover genuine epoch85 before attempting the selected-best image path or default continuation.\n", 'zeromrg_checkpoint_recovery.md': '# ZeroMRG Gate 1 checkpoint recovery\n\n> **Gate 2 verification, 2026-10-11 IST:** the approved preflight loaded the actual Kaggle-mounted epoch90 checkpoint and computed SHA-256 `ecc204f031b0fa83b4734c3228f20ba2172ce291288f31a6551921126546c299`, matching the local canonical artifact. GPU decoder/112-buffer SGD restoration and disposable save/restore passed. Epoch85 best weights remain missing; the guard correctly rejected epoch8 without resetting history. Earlier statements about unexecuted GPU checks/remote latest identity describe the historical Gate 1 evidence. See [Gate 2 record](zeromrg_gate2_preflight.md); no full remote latest download or training is claimed.\n\nAudit date: 2026-10-10. Workspace: `E:\\Insha Project\\ZeroMRG`. Starting evidence: [Gate 0](kaggle_gate0_assessment.md); its selected-best assertion is corrected here by inspecting the actual tensors and metadata. Scope is COV-CTR, 0% paired. No training, local real-model forward, remote execution, upload, or deletion occurred in this audit.\n\n## Decision and evidence\n\n**Epoch 90 is the canonical latest recovery checkpoint. The history-selected epoch-85 model is NOT recovered.** Model/SGD restoration passes; complete selected-best recovery is BLOCKED. An epoch number and BLEU score in history do not establish the existence of corresponding weights.\n\nMachine-readable evidence: [archive audit](evidence/gate1_checkpoint_audit.json), [independent remote best audit](evidence/gate1_remote_best_audit.json). The complete extracted backup is retained in ignored `artifacts/reconciliation/gate1_20261010_checkpoint_audit/epoch90/`. The separately downloaded remote best is in `artifacts/reconciliation/gate1_20261010_remote_best/`. Credentials and weights are excluded from commits.\n\n| Artifact | Verified metadata | Bytes | SHA-256 |\n|---|---|---:|---|\n| Downloads `zeromrg_covctr_training_backup (5).zip` | Canonical epoch-90 backup; ZIP CRC check passed, 15 members | 693642148 | `e3fc6449cfc8098ff43cefa9da704738d3f3898b3c17d7b3058caac5cec762e7` |\n| Extracted `text_decoder_last.pt` | epoch 90; full decoder, optimizer, history and selection counters | 459275387 | `ecc204f031b0fa83b4734c3228f20ba2172ce291288f31a6551921126546c299` |\n| Extracted `text_decoder_best.pt` | **epoch 8**, validation text BLEU-4 `52.33887482243904`, vocabulary 368, max tokens 160 | 229639049 | `dba64646e89f1c55c69c9e3a7361141829e4dc7ec32114f98f2edce8b89a0568` |\n| Remote `inshasadaf114/zeromrg-checkpoint-backup5-zip/text_decoder_best.pt` | **epoch 8**, same metadata and byte-identical to extracted best | 229639049 | `dba64646e89f1c55c69c9e3a7361141829e4dc7ec32114f98f2edce8b89a0568` |\n| `text_training_history.json` | 90 contiguous epochs; maximum at epoch 85 | See evidence JSON | `ccd65f2ec0ce2f569f683233f5a13c48c26bb27075aa9661bc7a272c3a14a015` |\n\nThe independent remote best has exactly the keys `decoder`, `epoch`, `max_tokens`, `validation_bleu4`, `vocabulary_size`; 112 float32 decoder tensors totaling 57,399,152 parameters. It has **no optimizer, scheduler, history or RNG state**. Its weights are architecturally valid, but it fails validation-selected best matching. No epoch-85 model was found in the accessible, bounded search.\n\n## Search coverage and limits\n\nSix local Downloads archives were inspected, including the unnumbered backup and backups `(1)` through `(5)`. Their saved latest epochs are 15, 30, 45, 60, 75, 90. All latest checkpoints pass the model/SGD/history audit. Every archived best has epoch-8 metadata and the same member CRC; history selects epoch 8 for epochs 15/30, epoch 35 for 45/60/75, and epoch 85 for epoch 90. The canonical extracted best and independently retrieved remote best additionally have matching SHA-256 hashes.\n\nThe search covered existing repository artifacts/checkpoint filenames, Downloads backup files and notebooks, Downloads/Compressed, bounded Documents and Documents/Codex listings (depth two), and sibling project directory names. Repository CPU-smoke outputs are diagnostic artifacts from a different tiny configuration, not candidates for this trained decoder. Large raw-image trees, virtual environments, Git internals, entire disks, cloud storage, live interactive RAM and unavailable older remote versions were not searched. Desktop was unavailable. This is a bounded negative finding, not proof that no epoch-85 copy exists anywhere.\n\nRead-only authenticated Kaggle discovery found the owned backup5 dataset and refreshed workflow status. Current workflow and preprocessing kernel output file listings returned headers without checkpoint entries. Historical notebook output through epoch 120 is real log evidence, but supplies neither saved weights nor momentum. No persistent model-and-optimizer pair after epoch 90 was verified. The remote **latest** checkpoint was listed but not independently downloaded and hashed; matching name/size is not proof of identity with the local canonical latest. Only the remote **best** was independently downloaded in full and hash-verified.\n\n## Epoch-90 restoration results\n\nSaved top-level keys are `epoch`, `decoder`, `optimizer`, `history`, `best_validation_bleu4`, `epochs_without_improvement`. History covers 1 through 90; saved best is epoch 85 with `56.44423922571333`; saved non-improvement counter is **5**, agreeing with history. Next epoch is **91**, derived from metadata.\n\nThe checkpoint-producing architecture is six post-norm Transformer decoder layers, width 768, eight attention heads, feed-forward width 3072, GELU/dropout 0.1, learned 160-position embeddings, vocabulary 368 and one conditioning memory token. Token embeddings are unscaled and there is no final decoder normalization. All 112 state keys, shapes, float32 dtypes and finite tensors match this implementation. The newer modular decoder defaults are incompatible and were not used for recovery.\n\nSGD has one parameter group: learning rate 0.001, momentum 0.99, dampening 0, weight decay 0, Nesterov false, maximize false; saved `foreach`/`fused` are null and `differentiable` false. All **112/112** parameters have correctly shaped, finite, nonzero momentum buffers. Strict CPU model loading and actual optimizer restoration preserved every saved momentum tensor exactly. Original source parameter order matches the recovered decoder; legacy state does not itself carry parameter names, so source reconciliation is necessary. Future checkpoints record names explicitly. There is no scheduler in the producing source or legacy state; this is consistent with the original constant-learning-rate experiment, not a missing scheduler to invent.\n\nNo RNG state, exact encoder revision, complete environment snapshot or complete configuration is stored in the legacy checkpoint. Matching model/optimizer state does **not** establish bitwise replay of the interrupted trajectory. A restarted dropout/data-order stream and package/hardware changes can produce different future results.\n\n## Selected-best discrepancy and recovery policy\n\nThe workflow constructs `text_decoder_best_slim.pt` in a separate cell, then packages that file under `text_decoder_best.pt`. The archived stale epoch-8 best across six backups is consistent with the slimming cell not being rerun after improvements. This is a plausible explanation, not verified evidence of exactly which interactive cells were executed.\n\nMissing epoch-85 weights do **not** prevent calculating the next SGD step or preserving the historical early-stopping comparison: the latest state includes the metric and counter. They **do** prevent returning the original selected best if subsequent epochs do not improve. They also prevent faithful image-to-report initialization from that selected checkpoint. Exact original stopping/model-selection recovery is therefore unavailable as a complete artifact pair.\n\nThe default continuation policy is fail closed: require a best artifact whose saved epoch and validation metric agree with history, then strictly check its architecture. Current inputs stop with `need epoch 85, found epoch 8`. Keep historical best `56.44423922571333` and patience `5/50`; never substitute epoch 90 or epoch 8, reset the metric, or fabricate weights. At unchanged best, 45 more non-improving epochs would stop at epoch 135. The 200-epoch ceiling is an inherited implementation assumption, not evidence that every run must reach 200.\n\nPreferred recovery: obtain a genuine epoch-85 artifact from another authorized backup/version, inspect its schema, finite tensor shapes and provenance, record its SHA-256, and verify its metric/epoch against history before enabling continuation or best-selected inference. Shape/metadata matching alone cannot prove provenance of an arbitrary file.\n\nIf recovery remains impossible, a **separately approved recovery experiment** could continue epoch-90 SGD while preserving the historical metric and patience and explicitly tracking `historical_selected_weights_missing`. Only a future strict improvement above the historical maximum would create a recoverable new winner. If stopping occurs without improvement, final selection remains blocked; neither latest nor best available may be relabeled as the historical winner. This exceptional policy is documented only: it is not enabled in the generated notebook and has not been approved. It would not recreate the lost epoch-85 model or prove the original trajectory.\n\n## Dependency integrity\n\nCanonical feature/split/vocabulary SHA-256 values are in the archive audit and embedded in both notebooks. Key dependencies:\n\n| File | SHA-256 |\n|---|---|\n| `train_manifest.csv` | `8902b8fea1163429aec6940a9744f7c792d1b7f9a00ca300a1fab17448ead973` |\n| `validation_manifest.csv` | `94c555cb25a19485616065079139722bfe7122a17639267ba5e03d9e4cfa7c5d` |\n| `test_manifest.csv` | `fcbcd1bd12220b0a3a286c4b1a82a8543e34c0a3495c5d0b8c89c81033b3eb66` |\n| `prompt_manifest.csv` | `f60c45a1dc9beecba16917d767f1cf6b0f67231141a9226efdb771a27f9281de` |\n| `decoder_vocabulary.json` | `c0e169c3558fe0cbf2e36431281aa223f64e2e4fb88f7361e23b33e12bcc3ad5` |\n| `mclip_text_features.pt` | `7275ebb5ea1a9ef273eee38b16e45fe6a3f42f154bb844ca2ba890e09853863a` |\n| `mclip_prompt_bank.pt` | `3cb770a2a470c70d95952828cf23abc088250571897b960a2a39e9bae6c8e1ce` |\n\nSaved dependencies verify 714 usable pairs, splits 571/71/72, 500 unique training-only prompts, vocabulary 368 and 768-dimensional finite features. Runtime assertions also check raw 726 CSV rows and 746 physical images on Kaggle; those actual mounts have not been executed in this Gate 1 session. Cached normalized prompt/text vectors differ slightly (maximum absolute difference `0.0018524825573`, minimum cosine `0.99988132715`); cause and exact historical encoder revision remain UNVERIFIED. Preserve the caches; do not silently re-encode or normalize the raw prompt bank differently.\n', 'zeromrg_source_reconciliation.md': "# ZeroMRG Gate 1 source reconciliation\n\nDate: 2026-10-10. Repository `E:\\Insha Project\\ZeroMRG`, Git HEAD `fd2a370a9c3fd4fe427a4f94ae3ea480ed8e315f`. The available starting assessment is `docs/kaggle_gate0_assessment.md`; the requested literal `(1)` copy was not found in the bounded inventory. Existing project instructions, research/docs, supplied base paper, model/evaluation/configuration/loading code and Kaggle notebooks were inspected before changes. The current owner's scope overrides older local-GPU, paired and IU-Xray planning: GPU work belongs on Kaggle and only COV-CTR 0% paired is active.\n\n## Sources retained without overwriting\n\nThe authenticated existing `scripts/kaggle.ps1` wrapper retrieved current `inshasadaf114/zeromrg-workflow` source and metadata into a new ignored directory `artifacts/reconciliation/gate1_20261010_remote/`. Original notebooks remain unchanged. SHA-256 values identify inspected source snapshots, not verified execution of every cell.\n\n| Source | Cells | SHA-256 |\n|---|---:|---|\n| Current remote `zeromrg-workflow.ipynb` | 55 | `b6589232df1ff75e216c59a76d2b36308c85e4086c05f505ee34d161d84424ad` |\n| Previously downloaded `C:\\Users\\Insha sadaf\\Downloads\\zeromrg-workflow.ipynb` | 49 | `ec47e576c72c0d908ec67c23358ad85549ff3e2189e70965e0fde3e3d584b563` |\n| `notebooks/ZeroMRG_COV_CTR_End_to_End_Kaggle.ipynb` | 28 | `992e5d7f5cd218d2fba86a05f23c7d872d52fea26b5d04535761fe50bfdc5056` |\n| `notebooks/ZeroMRG_COV_CTR_Resume_Epoch95_Kaggle.ipynb` | 22 | `846fcd8ad5c5d27e001697387bde5daeb686371b709e1c415510831da1651a55` |\n\nThe two workflow snapshots share 43 identical code cells. The older snapshot has 44 code cells and 43 executed cells; its console logs extend to epoch 120. The newly pulled source has 46 code cells and no execution outputs/counts. Its additional cells include installation and an epoch-90 restoration tail; the old continuation ceiling 120 becomes 95. Neither log history nor a COMPLETE kernel status proves that a later checkpoint is persistent. Refreshed output listings contained no retrievable workflow checkpoint entries. Saved remote metadata attaches `inshasadaf114/dataset` and `inshasadaf114/zeromrg-checkpoint-backup5-zip`, private GPU with Internet enabled.\n\n## Material comparison and implications\n\n| Area | Actual checkpoint-producing workflow (old/current) | Repository end-to-end / legacy resume | Canonical Gate 1 decision |\n|---|---|---|---|\n| Dataset attachments and paths | Owner-qualified `/kaggle/input/datasets/inshasadaf114/dataset/COV-CTR with English reports`; backup5 attached in current metadata | End-to-end discovers input data; resume expects an epoch-95 archive/artifact and discovers candidates | Explicit user-selected checkpoint and raw root; same two attachments; assert mounts, never choose the largest epoch automatically |\n| Encoder identity/version | Metadata names `M-CLIP/XLM-Roberta-Large-Vit-L-14`, matching 768-D OpenAI CLIP ViT-L/14 visual tower; exact revision absent | Modular wrappers use these named encoders but do not pin historical revisions | Preserve cached features and their hashes; check companion visual tower; record new environment/weight provenance; historical revision UNVERIFIED |\n| Text encoder initialization | Both workflow exports call `text_model`/`text_tokenizer` but omit their initialization and `MCLIP_NAME` definition | End-to-end instantiates wrappers; direct package forward can have device-placement differences | Historical workflow is not self-contained for re-encoding; readiness notebooks use existing verified caches and require no text model session globals |\n| Decoder | Six layers, 768 width, eight heads, 3072 FFN, GELU, post-norm, dropout .1, learned positions160; unscaled token embeddings, no final norm | Modular decoder scales embeddings by sqrt(width), has a final norm by default and 512 positional capacity; legacy resume still assumes epoch95 | `scripts/zeromrg_gate1_runtime.py` implements the exact producing architecture and keys; strict restore, no shape adaptation or model substitution |\n| Checkpoint schema | Latest: epoch, decoder, SGD state, history, best metric and non-improvement counter; best: decoder/epoch/metric/vocabulary/length | Modular experiment/checkpoint conventions differ; legacy resume hard-codes minimum/source epoch95 | Inspect actual schema, state/parameter order, history and matching best; derive next epoch; missing/incompatible state stops clearly |\n| Embeddings | L2-normalized report conditions, width768; raw 500x768 prompt vectors; image raw output cast float32 | Modular prompt-bank path normalizes prompt rows, changing attention logits | Reuse raw saved prompt vectors and normalized report conditions without recomputation |\n| Vocabulary/tokenization | 368 entries, PAD/BOS/EOS/UNK 0/1/2/3, regex `\\w+|[^\\w\\s]`, sorted lexicographic ordinary tokens, 160-token limit | Modular vocabulary uses frequency ordering and an apostrophe-aware regex; these IDs cannot decode historical weights faithfully | Verify saved mapping by training-only reconstruction; never rebuild IDs under modular tokenizer defaults |\n| Prompt construction | 500 fixed training report IDs and saved tensor order | Modular construction is independently configured and normalized | Assert train membership, uniqueness, report correspondence, exact order, size and dependency hashes |\n| Training/validation loops | Float32 text autoencoding, teacher forcing, valid-token CE, gradient clipping5; greedy text reconstruction on validation71; COCO PTB/BLEU4 x100 | End-to-end orchestrates additional regimes and test evaluation; resume assumes95 | Preserve decoder-only objective and validation rule; no test generation/tuning; bounded five-epoch chunks, disabled by default |\n| SGD/scheduler | SGD .001, momentum .99, batch6; no scheduler | Generic modular configurations/checkpoint fields are not proof of historic settings | Restore complete optimizer state, including all112 buffers; explicitly no scheduler; reject unknown nonempty scheduler state |\n| Selection/early stopping | Strict BLEU4 improvement, patience50, initial ceiling200; later sequential loop limits120/95; stale slim-best packaging | Legacy guide claims best85 and recovery95 without artifact evidence | Retain saved best56.44423922571333/counter5; fail on best85-versus-file8; ceiling200/BLEU4 are documented inherited assumptions |\n| Zero-shot path | Frozen CLIP visual identity and artificial finite embedding probe are present; no complete image-to-report evaluation path in these workflow exports | End-to-end has a path, but automatically proceeds to held-out test and 10%/100% paired experiments | Separate one-training-image diagnostic, raw prompt enrichment equations, frozen models, greedy decoder; no paired/test execution |\n| Persistence | Last/best working files; separate slim-best cell then ZIP packaging can preserve stale weights; working directory is session-local | Resume backup logic does not establish external durability | Atomic latest every epoch; package the actual current selected-best file, full dependencies/history/config; immutable exports and independently retrieved hashes |\n| Execution safety | Earlier 200/95 training cells precede restore tail and can start fresh if a checkpoint is absent | End-to-end begins broad workflow; legacy resume rejects valid epoch90 | Use new private Gate 1 slugs, Kaggle-only guard, no silent scratch fallback, training flag false; no remote push/run performed |\n\n## Canonical implementation\n\nThe canonical **recovery architecture** is the historical Kaggle checkpoint-producing decoder, implemented in `scripts/zeromrg_gate1_runtime.py` and embedded verbatim in the two Gate 1 notebooks. Canonical dependencies are the verified epoch-90 backup manifests, vocabulary and encoder caches, not newly generated modular artifacts. The current pulled notebook is retained as evidence, not designated a safe runnable end-to-end source. Original notebooks are not silently patched or overwritten.\n\n`scripts/build_gate1_notebooks.py` maintains self-contained notebooks in `notebooks/gate1/preflight/` and `notebooks/gate1/continuation/`, each with metadata pointing to a new private slug. Its regeneration is idempotent for unchanged sources. Static validation checks embedded runtime integrity, pinned dependency hash literals, cell syntax, metadata, disabled training and absence of backward/optimizer steps in preflight. See [validation evidence](evidence/gate1_notebook_validation.json). Local tests verify small CPU state restoration and architecture behavior; the actual full checkpoint CPU restoration was checked separately without running a real decoder forward.\n\n## Zero-shot paper fidelity\n\nThe supplied `paper/2026 medical report generation.pdf` was inspected directly (text extraction retained under the reconciliation directory). Section 3 describes frozen MCLIP encoders, text-only decoder reconstruction and prompt enrichment; Section 4 provides SGD .001/momentum .99/batch6, 500 COV-CTR prompts and 50-epoch validation-BLEU patience. It does not specify the decoder internals, tokenizer, split seed, exact revisions, raw-versus-unit prompt preprocessing, greedy search or maximum sequence length. These remain implementation assumptions, not paper-stated specifications.\n\nThe named Hugging Face M-CLIP checkpoint is the **text** tower with an OpenAI CLIP ViT-L/14 companion image tower, sharing a 768-D projected space ([official model card](https://huggingface.co/M-CLIP/XLM-Roberta-Large-Vit-L-14)). Dimensions alone do not establish alignment for an arbitrary substituted vision model. Gate 1 uses the named companion tower with RGB/canonical CLIP 224x224 preprocessing, evaluation mode, all image/decoder gradients disabled, visual parameter dtype respected and output promoted to float32. No vision fine-tuning is introduced.\n\nFor image vector `v` and prompt rows `P`, paper Eq. 6 is `softmax(v P^T / sqrt(768)) P`; Eq. 5 is the L2-normalized residual `normalize(v + attended_prompts)`. The runtime implements precisely this scaled dot product, softmax over 500 prompts, residual and final L2 normalization. It does not replace the operation with cosine nearest-neighbor retrieval, generic text prompting, temperature tuning or separately normalized prompt rows. Raw prompts are an existing source assumption whose historical feature provenance is preserved by hashes.\n\nGreedy generation starts at BOS, takes the next-token argmax under the causal mask, pads completed examples and stops at EOS for all examples or length160 including BOS. Postprocessing removes PAD/BOS, stops at EOS and joins punctuation under the historical rule. It does not remove generated UNK by inventing text. Generation/length choice is documented because the paper is silent.\n\nFidelity limits: historical text initialization/freeze evidence is incomplete in exported source; metadata gives model name but not exact revision/package checksum. Normalized prompt vectors and cached report vectors are extremely close but not identical, so their exact common encoding run is unverified. The image-conditioned diagnostic has not run on Kaggle, and is presently blocked by missing selected-best weights. No zero-shot result or final evaluation is claimed. The modular normalized-prompt/scaled-decoder path must not be used with historical checkpoints.\n\n## Sequence and cost decision\n\n1. **Diagnostic preflight first, after explicit approval.** It checks mounts/dependencies/latest restoration and a finite training-text forward/save/restore. With current inputs it reports BLOCKED on best selection before downloading/loading CLIP. This is a short bounded GPU diagnostic, not a training run.\n2. Recover and hash/provenance-verify epoch85 if possible. Then complete the one-training-image frozen path diagnostic (Path B). This tests integration before expensive continuation; it is not a final test result and does not demonstrate training convergence.\n3. Full continuation (Path A) is permitted only after selected-best recovery or approval of the separately documented incomplete-selection recovery policy, and after explicit remote-training approval. Restore original SGD and early-stopping state. At unchanged best, 45 further epochs remain until patience exhausts; with improvements up to110 epochs remain under the inherited ceiling. Five-epoch exports reduce losses between durable backups. Actual wall time/GPU cost is unmeasured; do not invent estimates.\n4. Only after a scientifically defensible stopping/selection decision and a recoverable selected winner, approve final held-out 0% image-to-report evaluation. Never tune on the 72 test samples.\n\nPath B is substantially cheaper than multi-epoch Path A and exposes image integration failures early. Both preserve split separation when restricted as implemented. Training readiness does not establish that extra training is scientifically unnecessary: epoch90 has not met original stopping rules. Conversely, reaching200 is not itself required. Missing best weights are a selection blocker for both the default complete continuation and selected-best image inference, not a reason to substitute architectures or restart the reproduction.\n", 'zeromrg_checkpoint_persistence.md': "# ZeroMRG COV-CTR checkpoint persistence procedure\n\nDate: 2026-10-10. This procedure applies only to explicitly approved Kaggle execution of the 0%-paired experiment. No remote publication has been performed. `/kaggle/working` is writable session storage; the existence of a file there is not proof of durable recovery after an interrupted interactive session.\n\n## Artifact responsibilities\n\n| Artifact | Purpose and required contents |\n|---|---|\n| `text_decoder_last.pt` | Latest **resumable** state: completed epoch, full decoder, full SGD/momentum, explicit scheduler `None`, contiguous history, historical best metric and patience counter, architecture, parameter names/order, training configuration, dependency hashes, Python/Torch CPU/CUDA RNG states and provenance |\n| `text_decoder_best.pt` | Actual validation-selected decoder, epoch/metric, vocabulary and positional dimensions, architecture/dependency metadata; need not contain the latest optimizer |\n| History/configuration | Full per-epoch text-reconstruction losses and validation BLEU4; batch/optimizer/patience/ceiling, split and encoder assumptions, source/runtime/environment provenance |\n| Frozen dependencies | Exact vocabulary, train/validation/test/validated/prompt manifests, validation audit, normalized text features and raw prompt bank; immutable files with SHA-256 |\n| Evaluation artifacts | Separate stage-labelled image-to-report predictions/references, metric settings/scores, selected checkpoint hash, image/text encoder weight revisions, environment and execution provenance. No final evaluation artifacts exist yet |\n\nCurrent legacy last state lacks RNG/configuration fields; this limitation is retained in provenance when migrating. The selected epoch85 is missing. Do not package the stale epoch8 file as a valid epoch85 best or reset selection history to make the package look complete.\n\n## Every completed epoch, when training is approved\n\n1. Keep the source dataset and attached backup immutable. Use a unique `/kaggle/working/zeromrg_gate1_<id>` output directory.\n2. Evaluate the original validation text-reconstruction metric. On strict improvement only, atomically save the actual current best weights with their matching epoch/metric. Preserve prior best on ties or deterioration.\n3. Validate latest state and selected-best metadata as a pair. Atomically save the latest full model/optimizer state **every epoch**, using a same-directory temporary file, flush/fsync and replace. Save contiguous history/configuration. RNG is captured after the epoch's validation; future restore replays saved RNG immediately before training, after diagnostics.\n4. The latest checkpoint embeds history, so a crash between its atomic replacement and the external history write is detectable. Current notebooks refuse an external-history mismatch; reconcile the external file against the embedded checkpoint in a documented local recovery step instead of guessing an epoch. Latest and best are separate files, not a transactional pair: an interrupted improvement may similarly require selecting an intact earlier immutable bundle. Never silently combine conflicting versions.\n5. Export an immutable epoch-labelled bundle after each five-epoch interval, at the end of the at-most-five-epoch session chunk, or on early stopping. The notebook copies the **current selected-best file**, not a stale separately generated slim copy. Include latest, best, history, configuration, all required immutable dependencies, the embedded runtime and per-file SHA-256 manifest. Produce ZIP and external ZIP SHA-256 sidecar. Check the package's storage size against available Kaggle disk before the next chunk; multiple immutable snapshots consume space. No existing checkpoint is deleted by this code.\n\nNew checkpoints capture the stochastic sources used by this decoder-only loop (Python and Torch CPU/CUDA, with the data-loader epoch seed explicit). This is not a promise of bitwise replay across hardware/PyTorch changes. Historical encoder features are reused and no stochastic NumPy processing is used in this loop. Record any future additional RNG dependency before adding it.\n\n## Durability outside the session\n\nThe training chunk returns after at most five epochs; do not begin another chunk until the export is externally secured. An export message explicitly says `DURABILITY UNVERIFIED`.\n\n**Practical local route:** download the immutable ZIP plus sidecar to a new non-overwriting backup location. Verify ZIP SHA-256, CRC/member hashes, load metadata with `weights_only=True`, inspect model/SGD/history and require the selected-best epoch/metric match. Log the recovered epoch, best epoch, file sizes/hash and storage location. This proves recovery from that retrieved copy, not availability of an untested remote version.\n\n**Private Kaggle route (requires explicit approval):** prepare a clean dataset staging directory for an epoch-labelled private checkpoint dataset/version. Include only the audited bundle/manifest/configuration; never credentials. Use the installed wrapper's `datasets create` or `datasets version` interface after checking its help and reviewing exact staging metadata. This publishes remote artifacts and is not authorized by Gate 1 preparation. Existing remote datasets must not be overwritten without approval. Optionally save a private committed kernel version with outputs; this is also a remote execution/persistence action needing approval.\n\nAfter publication, list the exact private dataset/version and independently download its backup into another new directory. Compare the downloaded ZIP and member hashes and re-run model/optimizer/best/history verification. Only then mark that version **retrievable and verified** in the recovery record. Listing success or an upload response alone is insufficient. Attach that exact audited version before another training chunk; verify the mounted hashes again.\n\nKaggle CLI `kernels push` executes a new version; it is not a harmless source-only upload. The installed CLI help has no `--no-run` option. Use [official kernel documentation](https://github.com/Kaggle/kaggle-cli/blob/main/docs/kernels.md) and the installed command help for any execution settings; no push, Save Version, dataset publication or training is part of this completed local task.\n\n## Current durable evidence\n\nThe local epoch90 archive and extracted dependencies are verified. The independently downloaded remote best file is retrievable, hash-identical and **stale epoch8**. A remotely retrieved epoch90 latest pair, genuine epoch85 best, or any later persistent checkpoint has not been verified during Gate 1. Do not label the proposed export strategy implemented-and-durable until a corresponding future export is actually retrieved and verified.\n", 'zeromrg_gate2_preflight.md': "# ZeroMRG Gate 2 bounded Kaggle preflight\n\nExecuted after explicit owner approval of one preflight only. Submission date: 2026-10-10 UTC; results collected 2026-10-11 Asia/Calcutta. Scope: COV-CTR 0%-paired, no decoder training, continuation, final zero-shot inference, test scoring or checkpoint overwriting.\n\n**Gate 2: BLOCKED by missing validation-selected epoch 85 weights. The bounded diagnostic behaved as designed.** Kaggle version 1 ended with `KernelWorkerStatus.ERROR` because the notebook deliberately raises after writing its BLOCKED report. Latest decoder restoration, optimizer restoration and GPU forward checks passed.\n\n## Submission and safety evidence\n\nRead [Gate 1 readiness](zeromrg_gate1_readiness.md) and [checkpoint recovery](zeromrg_checkpoint_recovery.md) before execution. The unchanged preflight SHA-256 was `3d9731b0e36bf0aa0ab591f3dfa370aabeb6512f0670d6201c141f0ca7b54991`. Static cell syntax, embedded source and approved attachment metadata were checked; the notebook contains no training loop, backward call, optimizer step or held-out scoring. It reads checkpoint inputs from immutable mounted datasets and writes only a unique diagnostic output directory. Missing selected-best weights cause a report and exception before CLIP import/loading.\n\nThe complete nine-entry owned-kernel listing showed no existing preflight target. The authorized command created a new kernel version1:\n\n```powershell\n.\\scripts\\kaggle.ps1 -KaggleArgs @('kernels','push','-p','notebooks/gate1/preflight','-t','1200')\n```\n\nThe installed client forwards `is_private`, dataset attachments and the 1200-second session timeout. After submission, independently pulled server metadata verified **private**, GPU enabled, TPU disabled, and exactly `inshasadaf114/dataset` plus `inshasadaf114/zeromrg-checkpoint-backup5-zip`. Retrieved cell source matched the audited submission; the pulled JSON byte hash differs due to server serialization: `dd0946729f6b07c353f589699bc98478fe88ed506fab3f72ca1aab5414e6be18`.\n\nTarget: [private preflight](https://www.kaggle.com/code/inshasadaf114/zeromrg-covctr-gate1-preflight). It initially queued for roughly 23 minutes before RUNNING was observed at 18:33:52 UTC. The collected execution log's last elapsed timestamp is approximately 47.03 seconds; billed GPU time was not measured. Terminal ERROR status was confirmed after output retrieval. No second version, continuation kernel, dataset mutation or Git push was submitted.\n\n## Actual retrieved results\n\n| Check | Result and observed evidence |\n|---|---|\n| Mounted paths | PASS: `/kaggle/input/datasets`; exact checkpoint and COV-CTR raw paths resolved |\n| Raw population | PASS: 726 report rows, 746 physical images; duplicate-ID assertion passed |\n| Saved population/splits | PASS: 714 usable pairs, 571 train / 71 validation / 72 test; original split reused, no regeneration |\n| Dependencies/provenance | PASS: pinned hashes, report/image correspondence, training-only vocabulary 368 and 500 ordered training prompts; 768-D text/prompt dimensions |\n| Latest load | PASS: epoch 90; next 91 derived; checkpoint SHA matches canonical local latest |\n| Decoder compatibility | PASS: actual historical six-layer post-norm decoder, width 768 / eight heads / FFN 3072, 160 positions, unscaled embeddings / no final norm; 57,399,152 parameters |\n| Optimizer/momentum | PASS: complete SGD restoration; 112/112 finite nonzero momentum buffers; restored buffers equal saved values; no optimizer update |\n| History/selection state | Preserved: best history epoch 85, BLEU4 `56.44423922571333`, non-improvement counter 5; no metric/patience reset |\n| GPU/memory | Tesla T4; 14.45977783203125 GiB free of 14.56219482421875 GiB **before decoder loading**; peak and post-load free memory were not measured |\n| Finite forward | PASS: one fixed training report; CE `0.08689973503351212`; finite outputs, no gradients or backward/update. This is a diagnostic loss, not an image-generation or test metric |\n| Output writability | PASS: write/read probe; retrieved `write_probe.txt` contains `writable` |\n| Disposable save/restore | PASS: isolated full-state serialization/reload, exact weights/momentum, identical restored logits; disposable artifact removed by its own temporary-directory context |\n| Best checkpoint | BLOCKED: history requires 85 but saved file identifies 8; no substitution |\n| Frozen image/zero-shot path | NOT REACHED: stopped before CLIP import, model download or image generation |\n\nCanonical checkpoint SHA-256 computed on the **actual Kaggle mount** is `ecc204f031b0fa83b4734c3228f20ba2172ce291288f31a6551921126546c299`. This now establishes identity with the verified local latest through executed, retrieved diagnostics; it is not a claim of independently downloading the remote459MB latest file.\n\nDisposable diagnostic SHA-256: `df7aa91ad241a380ae37a0a89c02f349450589970f6a31802ef16bb30b5dbe0b`. It is not a new durable recovery checkpoint; the disposable file no longer exists. The original source checkpoints were never written. Canonical local last/best/history hashes were rechecked and remain byte-identical to Gate 1.\n\n## Exception and environment\n\nThe report records `Best checkpoint disagrees with history: need epoch 85, found epoch 8. Recover matching weights; do not substitute latest weights.` The intended terminal exception is:\n\n```text\nValueError: PREFLIGHT BLOCKED: validation-selected best weights unavailable. Latest restoration checks passed.\n```\n\nPapermill wraps this in `PapermillExecutionError` at input6, yielding Kaggle ERROR. Debugger frozen-module warnings and nbconvert/mistune escape-sequence warnings also appear; they did not prevent the preceding checks. No timeout, out-of-memory or earlier restoration exception appears in the retrieved execution log.\n\nObserved environment: Python 3.13.15, PyTorch 2.11.0+cu128, CUDA 12.8, NumPy 2.1.3, pandas 2.3.3, Pillow 12.3.0, transformers 5.16.1. `clip`, `multilingual-clip`, and `pycocoevalcap` were **not installed**. They were not needed for the executed cached-text diagnostic. The first is needed for the image path; pycocoevalcap/Java are needed for continuation validation. Java availability was not checked. No dependency installation occurred. Historical PyTorch 2.10 and absent original RNG/encoder revisions still limit exact trajectory reproduction despite this successful restoration.\n\n## Retrievable evidence and interpretation\n\n[Original preflight report](evidence/gate2_preflight_report.json), [original execution log](evidence/gate2_execution_log.json), and [audit summary](evidence/gate2_preflight_audit.json) are retained as small local evidence. Raw submission/status/source/output snapshots are in ignored `artifacts/gate2/preflight_20261010T181029719Z`.\n\nThe terminal `kernels files --csv` request returned only a header. Nevertheless, `kernels output` actually retrieved the report, write probe and log. Artifact retrieval, not the empty listing, is the positive evidence here. Report SHA-256 is `607b4d92d49698eaf69f4d8d874ef88ccc63f7c3db130412abcbd225341b0ca2`; log SHA-256 is `a4e30bc0f559f4b5c2adc82a7700e6112b9b666bb0a7fa30401a0726699e26c5`. Successful retrieval of diagnostics does not establish durability of any newly trained checkpoint; no training occurred and no new recovery checkpoint was exported.\n\nThis discrepancy also limits negative conclusions from earlier header-only kernel file listings. A future authorized checkpoint search should test actual output retrieval as well as listings before declaring a remote output absent. No other kernel's outputs were retrieved in this Gate 2 task, and no new epoch85 weights have been verified.\n\nOperational latest-state preflight passed and the selected-best guard worked. Overall readiness remains BLOCKED: recover and verify the provenance of genuine epoch 85 weights before default continuation or the selected-best image diagnostic. Preserve historical best/patience. A separately approved incomplete-selection policy remains only a documented option; it was not enabled. No final test metrics or valid selected-best zero-shot result are available.\n\nThe authorized preflight is complete. **No further remote execution or training is approved or automatic.** Do not rerun this notebook, install packages remotely, push continuation, or enable training from these results without a new explicit authorization.\n", 'gate2_preflight_report.json': '{\n  "kind": "preflight_only",\n  "checkpoint_sha256": "ecc204f031b0fa83b4734c3228f20ba2172ce291288f31a6551921126546c299",\n  "latest": {\n    "epoch": 90,\n    "next_epoch": 91,\n    "best_epoch": 85,\n    "best_validation_bleu4": 56.44423922571333,\n    "epochs_without_improvement": 5,\n    "momentum_buffers": 112,\n    "nonzero_momentum_buffers": 112,\n    "parameter_count": 57399152,\n    "legacy_rng_missing": true,\n    "scheduler": "none",\n    "compatibility": "PASS"\n  },\n  "dependency_assertions": "PASS",\n  "finite_train_forward_loss": 0.08689973503351212,\n  "disposable_restore": "PASS",\n  "versions": {\n    "python": "3.13.15",\n    "torch": "2.11.0+cu128",\n    "cuda": "12.8",\n    "numpy": "2.1.3",\n    "pandas": "2.3.3",\n    "Pillow": "12.3.0",\n    "clip": "not installed",\n    "multilingual-clip": "not installed",\n    "transformers": "5.16.1",\n    "pycocoevalcap": "not installed"\n  },\n  "gpu": "Tesla T4",\n  "zero_shot": "BLOCKED",\n  "overall": "BLOCKED",\n  "best_selection": "Best checkpoint disagrees with history: need epoch 85, found epoch 8. Recover matching weights; do not substitute latest weights."\n}', 'gate2_preflight_audit.json': '{\n  "observed_utc": "2026-10-10 18:56:05 UTC",\n  "running_observed_utc": "2026-10-10 18:33:52 UTC",\n  "terminal_status_confirmed_utc": "2026-10-10 18:52:28 UTC",\n  "local_date": "2026-10-11 (Asia/Calcutta); submission 2026-10-10 UTC",\n  "target": "inshasadaf114/zeromrg-covctr-gate1-preflight",\n  "version": 1,\n  "server_status": "KernelWorkerStatus.ERROR",\n  "authorization": "One bounded preflight only, timeout 1200 seconds; no continuation/training/test evaluation/checkpoint overwriting",\n  "gate2": "BLOCKED",\n  "diagnostic_behavior": "PASS: intended selected-best mismatch stop after latest diagnostics",\n  "submitted_notebook_sha256": "3d9731b0e36bf0aa0ab591f3dfa370aabeb6512f0670d6201c141f0ca7b54991",\n  "retrieved_remote_source_sha256": "dd0946729f6b07c353f589699bc98478fe88ed506fab3f72ca1aab5414e6be18",\n  "server_private": true,\n  "server_datasets": [\n    "inshasadaf114/dataset",\n    "inshasadaf114/zeromrg-checkpoint-backup5-zip"\n  ],\n  "dataset_counts": {\n    "raw_csv_rows": 726,\n    "physical_images": 746,\n    "usable": 714,\n    "train": 571,\n    "validation": 71,\n    "test": 72,\n    "prompts": 500,\n    "vocabulary": 368,\n    "embedding_dim": 768\n  },\n  "latest_checkpoint": {\n    "epoch": 90,\n    "next_epoch": 91,\n    "sha256": "ecc204f031b0fa83b4734c3228f20ba2172ce291288f31a6551921126546c299",\n    "model_parameters": 57399152,\n    "momentum_buffers": 112,\n    "finite_nonzero_momentum_buffers": 112,\n    "best_epoch_from_history": 85,\n    "best_validation_bleu4": 56.44423922571333,\n    "epochs_without_improvement": 5,\n    "legacy_rng_missing": true,\n    "compatibility": "PASS"\n  },\n  "gpu": {\n    "name": "Tesla T4",\n    "free_GiB_before_decoder_load": 14.45977783203125,\n    "total_GiB": 14.56219482421875\n  },\n  "finite_training_sample_forward_CE": 0.08689973503351212,\n  "disposable_restore": "PASS; model/momentum exact and restored logits equal",\n  "disposable_sha256": "df7aa91ad241a380ae37a0a89c02f349450589970f6a31802ef16bb30b5dbe0b",\n  "best_validation": "BLOCKED: need epoch 85, found epoch 8",\n  "exception": "ValueError: PREFLIGHT BLOCKED: validation-selected best weights unavailable. Latest restoration checks passed.",\n  "image_encoder_or_zero_shot_generation": "NOT REACHED",\n  "missing_packages_in_run": [\n    "clip",\n    "multilingual-clip",\n    "pycocoevalcap"\n  ],\n  "runtime_environment": {\n    "python": "3.13.15",\n    "torch": "2.11.0+cu128",\n    "cuda": "12.8"\n  },\n  "log_last_elapsed_seconds": 47.029258016,\n  "kernel_output_listing_caveat": "kernels files returned only a CSV header, but kernels output independently retrieved report, write probe, and log",\n  "raw_evidence_directory": "artifacts/gate2/preflight_20261010T181029719Z",\n  "report": "docs/evidence/gate2_preflight_report.json",\n  "execution_log": "docs/evidence/gate2_execution_log.json",\n  "report_sha256": "607b4d92d49698eaf69f4d8d874ef88ccc63f7c3db130412abcbd225341b0ca2",\n  "log_sha256": "a4e30bc0f559f4b5c2adc82a7700e6112b9b666bb0a7fa30401a0726699e26c5",\n  "canonical_local_checkpoints_and_history_unchanged": true,\n  "no_dataset_mutation_commands": true,\n  "no_continuation_submission": true,\n  "no_training_or_test_scoring": true\n}\n'}
HISTORICAL_REPORT_HASHES = {'zeromrg_gate1_readiness.md': '6d0eb0d9e6162390ccffe304d1b7c3f3fd327713cc4b3cfc119eab972fa46eed', 'zeromrg_checkpoint_recovery.md': 'f26a86f5ba6b2cbbc6e57cf64501083dc6458e202288103a71d0fcfc61806303', 'zeromrg_source_reconciliation.md': '9dd99f5aa26b5a7067c77d88fc4220ef5b86ef21cdf3448e5af1a0e4e051f8f4', 'zeromrg_checkpoint_persistence.md': '4346c6a9632e349c7f6938c8b1b18de239a4bac9ba95b0f22b4b34922834e659', 'zeromrg_gate2_preflight.md': 'f6c413a3d13ca8e01827096978d687d65b51117297e777a0196cfbdef23557d4', 'gate2_preflight_report.json': '607b4d92d49698eaf69f4d8d874ef88ccc63f7c3db130412abcbd225341b0ca2', 'gate2_preflight_audit.json': 'b7dcf44b2b5e3886e6b7ddc02b346ee1a914c6042ce303ab54ce59eb6e9d24dd'}

In [ ]:
import subprocess
ENABLE_APPROVED_RECOVERY_TRAINING = True
MAX_NEW_EPOCHS = 5
# A future baseline-only execution must explicitly approve this isolated metric install.
INSTALL_LOCKED_BLEU_PACKAGE = True
BASELINE_ZIP_NAME = 'recovery_export_epoch_090_54468b70.zip'
BASELINE_ZIP_BYTES = 1382728028
BASELINE_ZIP_SHA256 = 'dec3cd3391e2c107a7c9ed6209997f00554cd47cb0940ca339a74446253680b1'
BASELINE_LATEST_SHA256 = 'f641f7ef7312e74b2c79c0b76f1920498999a90328f296f50d75950784a4e404'
BASELINE_BEST_SHA256 = '6166e36198737bfbe9e0c35f03f0b320c44e76f3e05068ef6299d74e9179c6b2'
AUTHORIZED_START_EPOCH = 91
AUTHORIZED_END_EPOCH = 95
SESSION_AUTHORIZATION = 'One private Kaggle recovery session, epochs91-95 inclusive only'

In [ ]:
import gc, platform, shutil, sys, tempfile, time, unicodedata, uuid, zipfile

# Exact existing baseline export, independently downloaded/verified before this run.
import stat
require(Path('/kaggle/input').is_dir(), 'Kaggle execution only')
candidates = list(Path('/kaggle/input').rglob(BASELINE_ZIP_NAME))
require(len(candidates) == 1, 'Expected exactly one attached baseline export with the approved filename')
BASELINE_INPUT_ZIP = candidates[0]
require(BASELINE_INPUT_ZIP.stat().st_size == BASELINE_ZIP_BYTES and
        sha256(BASELINE_INPUT_ZIP) == BASELINE_ZIP_SHA256, 'Attached recovery export differs from approved baseline')
INPUT_RECOVERY_DIR = Path('/kaggle/working') / ('verified_recovery_input_' + uuid.uuid4().hex[:12])
INPUT_RECOVERY_DIR.mkdir(exist_ok=False)
with zipfile.ZipFile(BASELINE_INPUT_ZIP) as bundle:
    entries = bundle.infolist()
    require(len(entries) == len(set(item.filename for item in entries)), 'Duplicate input archive paths')
    require(sum(item.file_size for item in entries) < 5 * 2**30, 'Input export exceeds audited size bound')
    require(shutil.disk_usage(INPUT_RECOVERY_DIR).free > sum(item.file_size for item in entries) + 2**30,
            'Insufficient space for safe input restoration')
    for item in entries:
        require((INPUT_RECOVERY_DIR / item.filename).resolve().is_relative_to(INPUT_RECOVERY_DIR.resolve()),
                'Unsafe input archive path')
        require(stat.S_IFMT(item.external_attr >> 16) != stat.S_IFLNK, 'Symlink input refused')
    require(bundle.testzip() is None, 'Input archive CRC failure')
    bundle.extractall(INPUT_RECOVERY_DIR)
input_manifest = json.loads((INPUT_RECOVERY_DIR / 'sha256_manifest.json').read_text())
for name, item in input_manifest['files'].items():
    require(sha256(INPUT_RECOVERY_DIR / name) == item['sha256'] and
            (INPUT_RECOVERY_DIR / name).stat().st_size == item['bytes'], 'Input manifest differs: ' + name)
require(sha256(INPUT_RECOVERY_DIR / 'recovery_latest.pt') == BASELINE_LATEST_SHA256 and
        sha256(INPUT_RECOVERY_DIR / 'recovery_best_epoch_090.pt') == BASELINE_BEST_SHA256,
        'Approved epoch90 latest/best weights are absent or changed')
print('Approved private baseline export attached and verified:', BASELINE_INPUT_ZIP)
import gc, platform, shutil, sys, tempfile, time, unicodedata, uuid, zipfile
from collections import Counter, defaultdict
from importlib import metadata

# Select an exact checkpoint; no automatic newest-file selection or epoch assumption.
CHECKPOINT_PATH = INPUT_RECOVERY_DIR / 'recovery_latest.pt'
ARTIFACT_DIR = Path('/kaggle/input/datasets/inshasadaf114/zeromrg-checkpoint-backup5-zip')
BEST_CHECKPOINT_PATH = ARTIFACT_DIR / 'text_decoder_best.pt'
DATA_ROOT = Path('/kaggle/input/datasets/inshasadaf114/dataset/COV-CTR with English reports')
RUN_ROOT = Path('/kaggle/working') / ('zeromrg_gate3_' + uuid.uuid4().hex[:12])
require(Path('/kaggle/input').is_dir() and Path('/kaggle/working').is_dir(), 'Execute this notebook on Kaggle only')
require(torch.cuda.is_available(), 'Enable a Kaggle GPU; no local CPU fallback')
DEVICE = torch.device('cuda:0')
require(CHECKPOINT_PATH.is_file(), 'Set CHECKPOINT_PATH to the exact mounted latest checkpoint')
require(DATA_ROOT.is_dir(), 'Set DATA_ROOT to the attached COV-CTR directory')
require(RUN_ROOT.resolve().is_relative_to(Path('/kaggle/working').resolve()), 'Output must stay in /kaggle/working')
RUN_ROOT.mkdir(exist_ok=False)
write_probe = RUN_ROOT / 'write_probe.txt'
write_probe.write_text('writable', encoding='utf-8')
require(write_probe.read_text(encoding='utf-8') == 'writable', 'Output write/read failed')
print('Mounted roots:', [str(p) for p in Path('/kaggle/input').iterdir()])
gpu = torch.cuda.get_device_properties(0)
free_memory, total_memory = torch.cuda.mem_get_info()
versions = {'python': platform.python_version(), 'torch': torch.__version__, 'cuda': torch.version.cuda}
for distribution in ('numpy', 'pandas', 'Pillow', 'clip', 'multilingual-clip', 'transformers', 'pycocoevalcap'):
    try:
        versions[distribution] = metadata.version(distribution)
    except metadata.PackageNotFoundError:
        versions[distribution] = 'not installed'
print('GPU:', gpu.name, '| free/total GiB:', free_memory/2**30, total_memory/2**30)
print('Versions:', versions)
print('Selected checkpoint:', CHECKPOINT_PATH, '| output:', RUN_ROOT)

In [ ]:
# Legacy recovery must match the audited dependencies exactly. New checkpoints carry
# the same dependency hashes; changed splits/features are refused.
checkpoint = load_checkpoint(CHECKPOINT_PATH)
expected_dependencies = checkpoint.get('dependency_hashes', CANONICAL_DEPENDENCY_HASHES)
artifacts = validate_artifacts(ARTIFACT_DIR, expected_dependencies)
records = artifacts['records']
vocabulary = artifacts['vocabulary']['vocabulary']
text_cache = artifacts['text']
prompt_cache = artifacts['prompt']
external_history = json.loads((ARTIFACT_DIR / 'text_training_history.json').read_text(encoding='utf-8'))
require(external_history == checkpoint.get('original_history', checkpoint.get('history')), 'Original history differs')

# Verify mounted raw population and saved target reports without rebuilding splits.
csv_candidates = list(DATA_ROOT.rglob('reports_ZH_EN.csv'))
require(len(csv_candidates) == 1, 'Expected one attached COV-CTR CSV')
with csv_candidates[0].open(encoding='utf-8-sig', newline='') as handle:
    raw_rows = list(csv.DictReader(handle))
require(len(raw_rows) == 726, 'Expected 726 raw report rows')
images = [p for p in DATA_ROOT.rglob('*') if p.is_file() and p.suffix.lower() in
          {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}]
require(len(images) == 746, 'Expected 746 physical image files')
raw_ids = Counter(row['image_id'].strip() for row in raw_rows)
require(sum(count > 1 for count in raw_ids.values()) == 6, 'Duplicate-ID audit differs')
raw_by_id = {row['image_id'].strip(): row for row in raw_rows if raw_ids[row['image_id'].strip()] == 1}
image_index = defaultdict(list)
for p in images:
    image_index[p.name.lower()].append(p)
for row in records['validated']:
    key = row['image_id']
    normalized = re.sub(r'\s+', ' ', unicodedata.normalize('NFC', raw_by_id[key]['reports_En'])).strip()
    require(normalized == row['report'], f'Saved report differs from raw source: {key}')
    require(len(image_index[key.lower()]) == 1, f'Image missing/ambiguous: {key}')
print('Dataset/split counts: 726 rows, 746 images, 714 usable; 571/71/72 partitions')
print('Training-only vocabulary: 368; prompt bank: 500 x 768 raw; normalized text: 571 + 71')

def token_batch(rows):
    sequences = [encode_report(row['report'], vocabulary) for row in rows]
    ids = torch.full((len(rows), max(map(len, sequences))), 0, dtype=torch.long, device=DEVICE)
    for index, sequence in enumerate(sequences):
        ids[index, :len(sequence)] = torch.tensor(sequence, device=DEVICE)
    return ids

def feature_batch(rows, name):
    return torch.stack([text_cache[name][row['image_id']] for row in rows]).to(DEVICE)

def decoder_loss(condition, ids):
    logits = decoder(condition, ids[:, :-1])
    return F.cross_entropy(logits.reshape(-1, 368), ids[:, 1:].reshape(-1),
                           ignore_index=0, reduction='sum') / ids[:, 1:].ne(0).sum()

In [ ]:
# Never install/upgrade Torch, encoders, or the root project requirements here.
require(platform.python_implementation() == 'CPython' and sys.version_info[:2] == (3, 13),
        'Reaudit dependencies for a different Python family')
require(platform.system() == 'Linux' and platform.machine() == 'x86_64', 'Expected Kaggle Linux x86_64')
require(str(torch.__version__).startswith('2.11.') and torch.version.cuda == '12.8', 'Reaudit changed Torch/CUDA image')
require(versions['numpy'] == '2.1.3', 'Reaudit changed NumPy image')
require(shutil.which('java') is not None, 'Java is missing; stop without altering the environment')
java_result = subprocess.run(['java', '-version'], capture_output=True, text=True, check=True)
java_version = (java_result.stderr + java_result.stdout).strip()
metric_target = RUN_ROOT / 'metric_packages'
lock_path = RUN_ROOT / 'requirements-bleu.lock'
lock_path.write_text(METRIC_LOCK, encoding='utf-8')
require(INSTALL_LOCKED_BLEU_PACKAGE is True, 'Approve the exact isolated metric installation before executing')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--target', str(metric_target),
                '--no-deps', '--require-hashes', '--only-binary=:all:', '-r', str(lock_path)], check=True)
sys.path.insert(0, str(metric_target))
# pycocotools is deliberately excluded: this isolated BLEU/PTB path never imports it.
# This is not an installation for arbitrary COCO metrics.
for relative, digest in METRIC_IMPLEMENTATION.items():
    require(sha256(metric_target / relative) == digest, 'BLEU/PTB implementation differs: ' + relative)
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
from pycocoevalcap.bleu.bleu import Bleu

def score_predictions(references, predictions):
    require(len(references) == len(predictions) > 0, 'Metric population mismatch')
    ptb = PTBTokenizer()
    refs = ptb.tokenize({i: [{'caption': text}] for i, text in enumerate(references)})
    preds = ptb.tokenize({i: [{'caption': text}] for i, text in enumerate(predictions)})
    score = 100.0 * float(Bleu(4).compute_score(refs, preds)[0][3])
    require(math.isfinite(score) and 0 <= score <= 100, 'Non-finite/out-of-range BLEU4')
    return score

toy = PTBTokenizer().tokenize({0: [{'caption': 'The lungs are clear . No pleural effusion .'}]})
require(toy[0] == ['the lungs are clear no pleural effusion'], 'PTB smoke test differs')
require(score_predictions(['the lungs are clear with no pleural effusion'],
                          ['the lungs are clear with no pleural effusion']) > 99.99, 'BLEU smoke test failed')
METRIC_FINGERPRINT = plain_metadata(dict(package='pycocoevalcap==1.2', wheel_sha256=METRIC_WHEEL_SHA256,
                                        implementation_sha256=METRIC_IMPLEMENTATION, java=java_version,
                                        tokenizer='Stanford PTB 3.4.1 lowercase punctuation removal',
                                        metric='corpus BLEU4 closest references x100'))
ENVIRONMENT = plain_metadata(dict(python=versions['python'], torch=str(torch.__version__),
                                  cuda=torch.version.cuda, numpy=versions['numpy'], gpu=gpu.name))
print('Locked BLEU/PTB smoke tests passed:', METRIC_FINGERPRINT)

In [ ]:
require(checkpoint.get('epoch') == 90 and checkpoint.get('schema_version') == RECOVERY_SCHEMA,
        'This authorization initializes only the reviewed full epoch90 recovery checkpoint')
require(checkpoint['original_stop_tracking'] == dict(best_epoch=85, best_validation_bleu4=56.44423922571333,
        epochs_without_improvement=5, patience=50), 'Historical starting patience/best differs')
require(checkpoint['recovery_best']['epoch'] == 90 and
        checkpoint['recovery_best']['validation_bleu4'] == 54.21140850351637 and
        checkpoint['recovery_best']['sha256'] == BASELINE_BEST_SHA256, 'Recovery baseline selection differs')
require(checkpoint['continuation_history'] == [], 'Starting checkpoint already contains training updates')
require(not ENABLE_APPROVED_RECOVERY_TRAINING or checkpoint.get('schema_version') == RECOVERY_SCHEMA,
        'Legacy90 initialization is baseline-only; training requires a retrieved and reviewed Gate3 checkpoint')
original_history = json.loads((ARTIFACT_DIR / 'text_training_history.json').read_text(encoding='utf-8'))
for name, digest in ORIGINAL_FILES.items():
    require(sha256(ARTIFACT_DIR / name) == digest, 'Historical source hash differs: ' + name)
historical_dir = RUN_ROOT / 'historical_evidence'
historical_dir.mkdir(exist_ok=False)
for name in ORIGINAL_FILES:
    destination = historical_dir / name
    shutil.copy2(ARTIFACT_DIR / name, destination)
    require(sha256(destination) == ORIGINAL_FILES[name], 'Historical copy integrity failure')
    destination.chmod(0o444)
for name, content in HISTORICAL_REPORTS.items():
    path = historical_dir / name
    path.write_bytes(content.encode('utf-8'))
    require(sha256(path) == HISTORICAL_REPORT_HASHES[name], 'Historical report copy differs')
    path.chmod(0o444)

validation_ids = [row['image_id'] for row in records['validation']]
validation_references = [row['report'] for row in records['validation']]
require(len(validation_ids) == 71, 'Validation population changed')
with torch.device('meta'):
    decoder = CheckpointReportDecoder()
validate_model_state(checkpoint, decoder)
decoder.load_state_dict(checkpoint['decoder'], strict=True, assign=True)

def write_json_exclusive(path, value):
    with Path(path).open('x', encoding='utf-8') as handle:
        json.dump(plain_metadata(value), handle, indent=2, allow_nan=False)

def measure_validation(epoch, stage):
    decoder.eval()
    predictions = []
    with torch.no_grad():
        for start in range(0, 71, 6):
            rows = records['validation'][start:start + 6]
            output = decoder.generate(feature_batch(rows, 'validation'), max_length=160)
            predictions.extend(decode_report(ids, vocabulary) for ids in output.cpu())
    require(len(predictions) == 71, 'Not all validation reports were generated')
    score = score_predictions(validation_references, predictions)
    filename = f'validation_epoch_{epoch:03d}.json'
    measured = dict(epoch=epoch, split='validation', image_ids=validation_ids,
                    references=validation_references, predictions=predictions,
                    validation_bleu4=score, metric_fingerprint_sha256=json_sha256(METRIC_FINGERPRINT))
    write_json_exclusive(RUN_ROOT / filename, measured)
    return dict(epoch=epoch, stage=stage, validation_bleu4=score, predictions_file=filename,
                predictions_sha256=sha256(RUN_ROOT / filename),
                metric_fingerprint_sha256=json_sha256(METRIC_FINGERPRINT))

def verify_saved_scores(payload):
    for row in payload['recovery_history']:
        value = json.loads((RUN_ROOT / row['predictions_file']).read_text(encoding='utf-8'))
        require(value['references'] == validation_references, 'Saved references differ from validation manifest')
        rescored = score_predictions(value['references'], value['predictions'])
        require(math.isclose(rescored, row['validation_bleu4'], rel_tol=0, abs_tol=1e-10),
                'Saved BLEU4 does not reproduce from saved validation predictions')

if checkpoint.get('schema_version') == RECOVERY_SCHEMA:
    require(checkpoint['metric_fingerprint'] == METRIC_FINGERPRINT, 'Metric drift; no automatic recovery reset')
    require(checkpoint['provenance']['environment'] == ENVIRONMENT, 'Environment drift; reauditing required')
    optimizer, recovery_checks = restore_recovery(checkpoint, decoder, original_history,
        CANONICAL_DEPENDENCY_HASHES, CHECKPOINT_PATH.parent, validation_ids, DEVICE)
    # Copy only authoritative references; orphan candidates are never auto-selected.
    names = {checkpoint['recovery_best']['file']} | {r['predictions_file'] for r in checkpoint['recovery_history']}
    for name in names:
        shutil.copy2(CHECKPOINT_PATH.parent / safe_name(name), RUN_ROOT / name)
    metadata_state = metadata_from_payload(checkpoint)
    selected_reference = checkpoint['recovery_best']
    payload = checkpoint
    verify_saved_scores(payload)
else:
    require(sha256(CHECKPOINT_PATH) == ORIGINAL_FILES['text_decoder_last.pt'],
            'Legacy initialization must use the exact canonical epoch90 file')
    optimizer, recovery_checks = restore_weights_optimizer(checkpoint, decoder, DEVICE)
    require(recovery_checks['epoch'] == 90 and recovery_checks['epochs_without_improvement'] == 5,
            'Original stopping initialization differs')
    seed_legacy_recovery()  # New documented seed, not reconstruction of missing historical RNG.
    baseline = measure_validation(90, 'baseline')  # Fresh measured score; NEVER copy history[89].
    metadata_state = new_recovery_metadata(original_history, baseline, uuid.uuid4().hex,
        CANONICAL_DEPENDENCY_HASHES, METRIC_FINGERPRINT,
        dict(environment=ENVIRONMENT, legacy_rng_missing=True, restart_seed=42,
             source_sha256=ORIGINAL_FILES['text_decoder_last.pt'], runtime_sha256=RUNTIME_HASHES,
             paired_regime='0%', dataset='COV-CTR'))
    best_path = RUN_ROOT / 'recovery_best_epoch_090.pt'
    exclusive_save(recovery_best_payload(decoder, baseline, metadata_state['recovery_id'],
                                        CANONICAL_DEPENDENCY_HASHES), best_path)
    baseline_best = load_checkpoint(best_path)
    require(all(torch.equal(baseline_best['decoder'][k], checkpoint['decoder'][k])
                for k in checkpoint['decoder']), 'Baseline weights differ from physical epoch90 source')
    del baseline_best
    selected_reference = best_reference(best_path, baseline)
    payload = latest_recovery_payload(metadata_state, decoder, optimizer, selected_reference)
    verify_saved_scores(payload)
    print('Measured epoch90 recovery baseline:', baseline['validation_bleu4'])
    print('Original epoch90 recorded score (comparison only):', original_history[-1]['validation_bleu4'])
    print('Original selected85 score remains:', ORIGINAL_RECORD['best_validation_bleu4'])

validate_recovery_files(payload, decoder, original_history, CANONICAL_DEPENDENCY_HASHES, RUN_ROOT, validation_ids)
exclusive_save(payload, RUN_ROOT / 'recovery_latest.pt')
reloaded = load_checkpoint(RUN_ROOT / 'recovery_latest.pt')
validate_recovery_files(reloaded, decoder, original_history, CANONICAL_DEPENDENCY_HASHES, RUN_ROOT, validation_ids)
require(all(torch.equal(payload['decoder'][k].cpu(), reloaded['decoder'][k]) for k in payload['decoder']),
        'New schema save/reload model integrity failure')
require(all(torch.equal(payload['optimizer']['state'][k]['momentum_buffer'].cpu(),
                        reloaded['optimizer']['state'][k]['momentum_buffer'])
            for k in payload['optimizer']['state']), 'New schema save/reload momentum integrity failure')
optimizer, recovery_checks = restore_recovery(reloaded, decoder, original_history,
    CANONICAL_DEPENDENCY_HASHES, RUN_ROOT, validation_ids, DEVICE)
resume_rng = reloaded['rng_state']
del checkpoint, reloaded, payload
gc.collect()
print('Recovery state verified:', recovery_checks)
print('Historical stopping:', metadata_state['original_stop_tracking'])
write_json_exclusive(RUN_ROOT / 'recovery_readiness_report.json', dict(
    status='BASELINE_OR_RESTORATION_VERIFIED; NO_TRAINING_YET', epoch=metadata_state['epoch'],
    report_scope='immutable initialization snapshot; final completed epoch is in recovery_latest/history',
    checks=recovery_checks, historical_record=ORIGINAL_RECORD,
    original_stop_tracking=metadata_state['original_stop_tracking'], recovery_best=selected_reference,
    dataset_counts=dict(train=571, validation=71, test=72), test_scoring=False,
    environment=ENVIRONMENT, gpu_memory_before_load_bytes=dict(free=free_memory, total=total_memory),
    metric_fingerprint=METRIC_FINGERPRINT, durable_retrieval_verified=False))

In [ ]:
def persist_epoch():
    payload = latest_recovery_payload(metadata_state, decoder, optimizer, selected_reference)
    validate_recovery_files(payload, decoder, original_history, CANONICAL_DEPENDENCY_HASHES, RUN_ROOT, validation_ids)
    # Only this newly created run's latest is replaced atomically. Input/legacy artifacts never change.
    atomic_save(payload, RUN_ROOT / 'recovery_latest.pt')
    atomic_json_path = RUN_ROOT / 'recovery_history.json.tmp'
    atomic_json_path.write_text(json.dumps(metadata_state, indent=2, allow_nan=False), encoding='utf-8')
    os.replace(atomic_json_path, RUN_ROOT / 'recovery_history.json')

def export_bundle():
    latest = load_checkpoint(RUN_ROOT / 'recovery_latest.pt')
    validate_recovery_files(latest, decoder, original_history, CANONICAL_DEPENDENCY_HASHES, RUN_ROOT, validation_ids)
    epoch = latest['epoch']
    del latest
    # Include historical evidence, all new-session winners, current winner, and all measurements.
    sources = [(p, 'historical_evidence/' + p.name) for p in historical_dir.iterdir()]
    sources += [(ARTIFACT_DIR / name, 'dependencies/' + name) for name in DEPENDENCIES]
    names = {'recovery_latest.pt', 'recovery_history.json', 'requirements-bleu.lock', 'configuration.json',
             'recovery_readiness_report.json'}
    names |= {p.name for p in RUN_ROOT.glob('recovery_best_epoch_*.pt')}
    names |= {r['predictions_file'] for r in metadata_state['recovery_history']}
    names |= {p.name for p in RUN_ROOT.glob('epoch_completion_*.json')}
    if (RUN_ROOT / 'training_session_report.json').exists():
        names.add('training_session_report.json')
    sources += [(RUN_ROOT / name, name) for name in sorted(names)]
    total_bytes = sum(p.stat().st_size for p, _ in sources)
    require(shutil.disk_usage(RUN_ROOT).free > total_bytes + 2**30,
            'Insufficient export disk space; latest is saved, stop and retrieve before another session')
    destination = RUN_ROOT / f'recovery_export_epoch_{epoch:03d}_{uuid.uuid4().hex[:8]}.zip'
    manifest = {relative: dict(sha256=sha256(path), bytes=path.stat().st_size) for path, relative in sources}
    with zipfile.ZipFile(destination, 'x', compression=zipfile.ZIP_STORED, allowZip64=True) as archive:
        for source, relative in sources:
            archive.write(source, relative)
        archive.writestr('scripts/zeromrg_gate1_runtime.py', RUNTIME_SOURCE)
        archive.writestr('scripts/zeromrg_gate3_runtime.py', RECOVERY_RUNTIME_SOURCE)
        archive.writestr('sha256_manifest.json', json.dumps(dict(files=manifest, runtime_hashes=RUNTIME_HASHES), indent=2))
    write_json_exclusive(destination.with_suffix('.sha256.json'), dict(file=destination.name, sha256=sha256(destination),
        bytes=destination.stat().st_size, epoch=epoch, durable_retrieval_verified=False))
    print('Export prepared; durability UNVERIFIED until independently retrieved:', destination)

write_json_exclusive(RUN_ROOT / 'configuration.json', dict(architecture=ARCHITECTURE, training=TRAINING,
    recovery_policy=RECOVERY_POLICY, historical_record=ORIGINAL_RECORD, runtime_hashes=RUNTIME_HASHES,
    dependency_hashes=CANONICAL_DEPENDENCY_HASHES, metric_fingerprint=METRIC_FINGERPRINT,
    environment=ENVIRONMENT, approved_training=ENABLE_APPROVED_RECOVERY_TRAINING, session_authorization=SESSION_AUTHORIZATION, authorized_start_epoch=AUTHORIZED_START_EPOCH, authorized_end_epoch=AUTHORIZED_END_EPOCH, baseline_input_zip_sha256=BASELINE_ZIP_SHA256, baseline_latest_sha256=BASELINE_LATEST_SHA256))
write_json_exclusive(RUN_ROOT / 'recovery_history.json', metadata_state)
export_bundle()  # Baseline/restart export BEFORE any optional training.

In [ ]:
def run_approved_training_chunk():
    global metadata_state, selected_reference
    require(ENABLE_APPROVED_RECOVERY_TRAINING is True, 'Training is disabled')
    require(type(MAX_NEW_EPOCHS) is int and 1 <= MAX_NEW_EPOCHS <= 5, 'Maximum five updates per approved session')
    start = metadata_state['epoch'] + 1
    require(start == AUTHORIZED_START_EPOCH == 91 and AUTHORIZED_END_EPOCH == 95 and MAX_NEW_EPOCHS == 5, 'This session is authorized only for epochs91-95')
    require(start == recovery_checks['next_epoch'], 'Already advanced; reload an explicit checkpoint before another chunk')
    require(start <= 200 and metadata_state['original_stop_tracking']['epochs_without_improvement'] < 50,
            'Original stopping condition met; no silent restart')
    require(not globals().get('_gate3_training_attempt_started', False),
            'Training already attempted in this kernel; restart from a complete verified checkpoint')
    # Persist across re-execution of this cell, including exceptions before a completed epoch.
    # Never retry with partly updated in-memory model/optimizer state.
    globals()['_gate3_training_attempt_started'] = True
    session_started = time.monotonic()
    torch.cuda.reset_peak_memory_stats()
    restore_recovery_rng(resume_rng)  # Reapply after diagnostics/export.
    for epoch in range(start, min(AUTHORIZED_END_EPOCH, 200, start + MAX_NEW_EPOCHS - 1) + 1):
        require(91 <= epoch <= 95, "Epoch outside authorized range")
        epoch_started = time.monotonic()
        decoder.train()
        generator = torch.Generator().manual_seed(42 + epoch)
        loader = torch.utils.data.DataLoader(records['train'], batch_size=6, shuffle=True,
            generator=generator, num_workers=0, collate_fn=lambda rows: rows)
        losses = []
        for rows in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = decoder_loss(feature_batch(rows, 'train'), token_batch(rows))
            require(bool(torch.isfinite(loss)), 'Non-finite decoder loss')
            loss.backward()
            torch.nn.utils.clip_grad_norm_(decoder.parameters(), 5.0, error_if_nonfinite=True)
            optimizer.step()
            losses.append(float(loss.detach().cpu()))
        measurement = measure_validation(epoch, 'continuation')
        updated, improved = append_recovery_epoch(metadata_state, epoch, sum(losses)/len(losses), measurement)
        if improved:
            candidate_path = RUN_ROOT / f'recovery_best_epoch_{epoch:03d}.pt'
            exclusive_save(recovery_best_payload(decoder, measurement, updated['recovery_id'],
                                                CANONICAL_DEPENDENCY_HASHES), candidate_path)
            selected_reference = best_reference(candidate_path, measurement)
        metadata_state = updated
        persist_epoch()  # EVERY completed epoch; model/optimizer/RNG/ledgers/current-winner hash.

        torch.cuda.synchronize()
        completion = dict(epoch=epoch, completion_status='COMPLETED_AND_LATEST_SAVED',
            training_loss=metadata_state['continuation_history'][-1]['training_loss'],
            validation_bleu4=measurement['validation_bleu4'], validation_count=71,
            immutable_historical_record=ORIGINAL_RECORD,
            original_stop_tracking=metadata_state['original_stop_tracking'],
            recovery_best=dict(selected_reference), learning_rate=optimizer.param_groups[0]['lr'],
            momentum=optimizer.param_groups[0]['momentum'], optimizer_updates_this_epoch=len(losses),
            epoch_elapsed_seconds=time.monotonic()-epoch_started,
            latest_file='recovery_latest.pt', latest_sha256=sha256(RUN_ROOT / 'recovery_latest.pt'),
            latest_bytes=(RUN_ROOT / 'recovery_latest.pt').stat().st_size,
            predictions_sha256=measurement['predictions_sha256'])
        write_json_exclusive(RUN_ROOT / f'epoch_completion_{epoch:03d}.json', completion)
        print('EPOCH_COMPLETED', json.dumps(completion, sort_keys=True))

        print('Recovery', epoch, 'validation reconstruction BLEU4', measurement['validation_bleu4'],
              'recoverable best', selected_reference, 'historical stopping', metadata_state['original_stop_tracking'])
        if metadata_state['original_stop_tracking']['epochs_without_improvement'] >= 50:
            break

    final_state = load_checkpoint(RUN_ROOT / 'recovery_latest.pt')
    validate_recovery_files(final_state, decoder, original_history, CANONICAL_DEPENDENCY_HASHES, RUN_ROOT, validation_ids)
    require(final_state['epoch'] <= AUTHORIZED_END_EPOCH, 'Authorization ceiling exceeded')
    verify_saved_scores(final_state)
    restored_optimizer, final_checks = restore_weights_optimizer(legacy_view(final_state), decoder, DEVICE)
    require(all(torch.equal(restored_optimizer.state[p]['momentum_buffer'], optimizer.state[p]['momentum_buffer'])
                for p in decoder.parameters()), 'Final optimizer restoration differs')
    del restored_optimizer
    completion_files = sorted(RUN_ROOT.glob('epoch_completion_*.json'))
    session_report = dict(status='TRAINING_CHUNK_COMPLETE_EXPORT_PENDING_RETRIEVAL',
        authorization=SESSION_AUTHORIZATION, start_epoch=AUTHORIZED_START_EPOCH,
        final_completed_epoch=final_state['epoch'], epoch_ceiling=AUTHORIZED_END_EPOCH,
        epoch_records=[json.loads(p.read_text()) for p in completion_files],
        original_historical_record=ORIGINAL_RECORD, original_stop_tracking=final_state['original_stop_tracking'],
        recovery_best=final_state['recovery_best'], environment=ENVIRONMENT,
        session_elapsed_seconds=time.monotonic()-session_started,
        gpu_peak_allocated_bytes=torch.cuda.max_memory_allocated(),
        gpu_peak_reserved_bytes=torch.cuda.max_memory_reserved(),
        full_state_restoration_checks=final_checks, no_image_inference=True, test_scoring=False,
        durable_retrieval_verified=False)
    write_json_exclusive(RUN_ROOT / 'training_session_report.json', session_report)
    del final_state
    export_bundle()
    print('AUTHORIZED_TRAINING_SESSION_COMPLETE', json.dumps(session_report, sort_keys=True))

    print('Chunk complete. Retrieve and verify export before another separately approved chunk.')

if ENABLE_APPROVED_RECOVERY_TRAINING:
    run_approved_training_chunk()
else:
    print('BASELINE/RESTORATION ONLY: no decoder updates; no images or held-out test scoring.')